# Graf-Kompass GraphRAG: Sozialbericht 2024

Dieses Notebook nutzt den Sozialbericht 2024 als breites daten- und gesellschaftsnahes Beispiel. Die Leitfrage ist bewusst fokussiert, damit Retrieval und Ontologie nicht in zu viele Themen gleichzeitig ausfransen.

**Leitfrage aktiv:** Was sagt der Sozialbericht 2024 über Armut und Teilhabe in Deutschland?

**Frage-Optionen:**
- leicht: Was sagt der Sozialbericht 2024 über Armut und Teilhabe in Deutschland?
- mittel: Wie hängen Einkommen, Bildung und Arbeit mit gesellschaftlicher Teilhabe zusammen?
- schwer: Wie hängen Einkommen, Armut, Bildung, Arbeit und gesellschaftliche Teilhabe im Sozialbericht 2024 zusammen, und welche Indikatoren belegen diese Lebenslagen?

**Done ist erreicht**, wenn `outputs/sozialbericht_2024/provenance.json`, `outputs/sozialbericht_2024/provenance.html`, `outputs/sozialbericht_2024/retrieval_brief.md`, `outputs/sozialbericht_2024/answer_rag_context.md`, `outputs/sozialbericht_2024/answer_graphrag_context.md`, `outputs/sozialbericht_2024/answer_llm_only.md`, `outputs/sozialbericht_2024/answer_rag_only.md`, `outputs/sozialbericht_2024/answer_graphrag.md` und `outputs/sozialbericht_2024/answer_comparison.md` entstehen.


## Lernkarte: Was du in diesem Notebook siehst

Dieses Notebook ist als praktische Lernstrecke gebaut. Es geht nicht darum, sofort ein produktives System zu bauen, sondern das Architektur-Pattern einmal mit den Händen zu verstehen.

Die vier Schichten sind:

1. **Dokumentgraph**: `Document -> Chunk`. Die Quelle wird in suchbare Textstücke zerlegt.
2. **Ontologiegraph**: `Ontology -> OntologyClass -> Entity/Concept -> ONTOLOGY_RELATION`. Die fachliche Kontextkarte wird als Graph geladen.
3. **Retrievalgraph**: `Question -> Vector Search -> Chunk -> Entity`. Die Frage findet ähnliche Chunks und hängt sie an Ontologie-Kontext.
4. **Antwortvergleich**: `Frage -> LLM-only`, `Chunks -> RAG-only`, `Chunks + Entitäten + Pfade -> GraphRAG`. So wird sichtbar, was der Graph gegenüber einfachem RAG zusätzlich leistet.

Hinweis: Das ist GraphRAG mit Neo4j/Cypher und providerfähigen Embeddings. Es ist kein GraphQL-Beispiel.


## 0. Vorbereitung

Dieser Abschnitt richtet die Notebook-Umgebung ein: Pfade, Konfiguration, Secrets aus `.env`, Modellparameter und die Szenario-Frage.

**Was der Code macht:** Er lädt Umgebungsvariablen, sucht optional die Neo4j-Aura-Credentials-Datei, setzt Modell- und Suchparameter und definiert die Quelle, die dieses Notebook verarbeiten soll.

**Was du danach sehen solltest:** Eine kurze Konfigurationsübersicht. Wichtig sind `source_patterns`, `output_dir`, `question`, `aura_credentials_loaded` und `neo4j_database`. Daran erkennst du, ob das Notebook mit dem richtigen Szenario arbeitet.

**Szenario-Fokus:** Lebensverhältnisse, Daten/Indikatoren, Einkommen, Armut, Bildung, Arbeit und Teilhabe.


In [ ]:
# Grundsetup: Imports, Projektpfade, Szenario-Konfiguration und kleine Anzeige-Helfer.
from __future__ import annotations

import fnmatch
import json
import os
import re
import textwrap
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import Markdown, display
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI
from pypdf import PdfReader

# Das Notebook soll sowohl aus dem Projektordner als auch aus notebooks/ heraus funktionieren.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
SCENARIO_ID = "sozialbericht_2024"
OUTPUT_DIR = ROOT / "outputs" / SCENARIO_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")


# Diese Anzeige-Helfer sorgen dafür, dass Notebook-Ausgaben lesbar bleiben.
def show_json(title: str, description: str, value: dict) -> None:
    formatted = json.dumps(value, indent=2, ensure_ascii=False)
    display(Markdown(f"### {title}\n\n{description}\n\n```json\n{formatted}\n```"))


def show_table(title: str, description: str, frame: pd.DataFrame) -> None:
    display(Markdown(f"### {title}\n\n{description}"))
    display(frame)


def show_file(title: str, description: str, path: Path) -> None:
    show_json(title, description, {"path": str(path)})


def load_key_value_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        separator = "=" if "=" in line else ":" if ":" in line else None
        if not separator:
            continue
        key, value = line.split(separator, 1)
        values[key.strip()] = value.strip().strip('"')
    return values


# Aura-Credentials können aus der heruntergeladenen Neo4j-Datei kommen; .env bleibt vorrangig.
def load_neo4j_aura_credentials(root: Path) -> dict[str, str]:
    files = sorted(root.glob("Neo4j-*-Created-*.txt"))
    if not files:
        return {}
    values = load_key_value_file(files[0])
    for key, value in values.items():
        if value and not os.environ.get(key):
            os.environ[key] = value
    if values.get("NEO4J_USERNAME") and not os.environ.get("NEO4J_USER"):
        os.environ["NEO4J_USER"] = values["NEO4J_USERNAME"]
    return values

AURA_CREDENTIALS = load_neo4j_aura_credentials(ROOT)

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai").strip().lower()
if LLM_PROVIDER not in {"openai", "github"}:
    raise ValueError("LLM_PROVIDER muss entweder 'openai' oder 'github' sein.")

GITHUB_MODELS_BASE_URL = os.getenv("GITHUB_MODELS_BASE_URL", "https://models.github.ai/inference")
GITHUB_MODELS_API_VERSION = os.getenv("GITHUB_MODELS_API_VERSION", "2022-11-28")
EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    os.getenv(
        "GITHUB_EMBEDDING_MODEL" if LLM_PROVIDER == "github" else "OPENAI_EMBEDDING_MODEL",
        "openai/text-embedding-3-small" if LLM_PROVIDER == "github" else "text-embedding-3-small",
    ),
)
ANSWER_MODEL = os.getenv(
    "ANSWER_MODEL",
    os.getenv(
        "GITHUB_ANSWER_MODEL" if LLM_PROVIDER == "github" else "OPENAI_ANSWER_MODEL",
        "openai/gpt-4o-mini" if LLM_PROVIDER == "github" else "gpt-4.1-mini",
    ),
)
ANSWER_TEMPERATURE = float(
    os.getenv("ANSWER_TEMPERATURE", os.getenv("OPENAI_ANSWER_TEMPERATURE", "0.2"))
)
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", os.getenv("OPENAI_EMBEDDING_DIM", "1536")))
TOP_K = int(os.getenv("TOP_K", "8"))
HOP_PENALTY = float(os.getenv("HOP_PENALTY", "0.08"))

QUESTION_OPTIONS = {
    "leicht": 'Was sagt der Sozialbericht 2024 über Armut und Teilhabe in Deutschland?',
    "mittel": 'Wie hängen Einkommen, Bildung und Arbeit mit gesellschaftlicher Teilhabe zusammen?',
    "schwer": 'Wie hängen Einkommen, Armut, Bildung, Arbeit und gesellschaftliche Teilhabe im Sozialbericht 2024 zusammen, und welche Indikatoren belegen diese Lebenslagen?',
}
ACTIVE_QUESTION_LEVEL = os.getenv(f"{SCENARIO_ID.upper()}_QUESTION_LEVEL", "leicht").lower()
DEFAULT_QUESTION = QUESTION_OPTIONS.get(ACTIVE_QUESTION_LEVEL, QUESTION_OPTIONS["leicht"])
QUESTION_ENV_KEY = f"{SCENARIO_ID.upper()}_QUESTION"
QUESTION = os.getenv(QUESTION_ENV_KEY, DEFAULT_QUESTION)
SOURCE_PATTERNS = ['Sozialbericht_2024_bf_k2.pdf']

show_json(
    "Konfiguration",
    "Diese Übersicht zeigt, mit welchem Projektordner, Modell, Suchlimit und welcher Quelle das Notebook gerade arbeitet.",
    {
        "root": str(ROOT),
        "data_dir": str(DATA_DIR),
        "provider": LLM_PROVIDER,
        "embedding_model": EMBEDDING_MODEL,
        "answer_model": ANSWER_MODEL,
        "answer_temperature": ANSWER_TEMPERATURE,
        "embedding_dim": EMBEDDING_DIM,
        "top_k": TOP_K,
        "question_env_key": QUESTION_ENV_KEY,
        "active_question_level": ACTIVE_QUESTION_LEVEL,
        "question": QUESTION,
        "question_options": QUESTION_OPTIONS,
        "source_patterns": SOURCE_PATTERNS,
    "output_dir": str(OUTPUT_DIR),
    "aura_credentials_loaded": bool(AURA_CREDENTIALS),
        "neo4j_database": os.getenv("NEO4J_DATABASE", "<default>"),
    },
)

## 1. Quellen laden und Chunking

Dieser Abschnitt macht aus der Quelldatei eine Liste kleiner Textstücke, die später gesucht und belegt werden können.

**Was der Code macht:** Er liest die konfigurierte PDF-, XML-, Markdown- oder Textquelle aus `data/`, extrahiert Text und zerlegt ihn in überlappende Wort-Chunks.

**Was du danach sehen solltest:** Eine kleine Tabelle mit Anzahl der geladenen Dokumente, Anzahl der Chunks und durchschnittlicher Chunk-Länge. Wenn hier `documents = 1` steht, wurde genau die Szenario-Quelle gefunden.

In [ ]:
# Quellen laden und in Text-Chunks zerlegen: aus PDFs, XML, Markdown oder Text wird ein einheitlicher Chunk-Strom.
def xml_text(element: ET.Element) -> str:
    return " ".join(part.strip() for part in element.itertext() if part and part.strip())


def load_xml_text(path: Path) -> str:
    tree = ET.parse(path)
    root = tree.getroot()
    chunks: list[str] = []

    # DocBook XML, wie das BSI-Kompendium, nutzt Namespaces. ElementTree liefert
    # Tags deshalb als {namespace}tag. local_name macht die Struktur lesbarer.
    for element in root.iter():
        local_name = element.tag.rsplit("}", 1)[-1]
        if local_name in {"title", "subtitle", "para", "simpara", "entry"}:
            text = xml_text(element)
            if text:
                chunks.append(text)

    if not chunks:
        chunks.append(xml_text(root))

    return "\n".join(dict.fromkeys(chunks))


def load_source(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if path.suffix.lower() == ".xml":
        return load_xml_text(path)
    return path.read_text(encoding="utf-8", errors="ignore")


# Überlappung verhindert, dass wichtige Aussagen genau an einer Chunk-Grenze verloren gehen.
def chunk_words(text: str, size: int = 850, overlap: int = 140) -> Iterable[str]:
    words = text.split()
    step = max(size - overlap, 1)
    for start in range(0, len(words), step):
        part = words[start : start + size]
        if part:
            yield " ".join(part)


# Einfache Rauschfilterung: Literatur-/URL-lastige Chunks sollen das Retrieval nicht dominieren.
def is_low_signal_chunk(text: str) -> bool:
    normalized = text.lower()
    bibliography_markers = ["doi", "http", "www.", " in: ", " et al.", "s. ", "isbn", "literatur", "verfügbar"]
    marker_count = sum(normalized.count(marker) for marker in bibliography_markers)
    climate_terms = ["klimaanpassung", "kommune", "kommunen", "maßnahme", "maßnahmen", "hebel", "resilienz"]
    has_domain_signal = any(term in normalized for term in climate_terms)
    return marker_count >= 8 and not has_domain_signal

# SOURCE_PATTERNS wählt pro Szenario bewusst nur die passende Demo-Quelle aus.
all_source_paths = sorted(
    p for p in DATA_DIR.glob("*")
    if p.suffix.lower() in {".pdf", ".md", ".txt", ".xml"}
)
source_paths = [
    p for p in all_source_paths
    if any(fnmatch.fnmatch(p.name, pattern) for pattern in SOURCE_PATTERNS)
]

if not source_paths:
    raise FileNotFoundError(f"Keine passende Quelle gefunden. Gesucht: {SOURCE_PATTERNS}. Vorhanden: {[p.name for p in all_source_paths]}")

# Ab hier entsteht das Dokumentmodell: Quelle -> Chunks -> später Embeddings und Graphknoten.
docs = [{"doc_id": p.name, "path": str(p), "text": load_source(p)} for p in source_paths]
raw_chunks = []
for doc in docs:
    for index, content in enumerate(chunk_words(doc["text"])):
        raw_chunks.append({
            "chunk_id": f"{doc['doc_id']}::c{index:03d}",
            "doc_id": doc["doc_id"],
            "content": content.strip(),
        })

chunks = [chunk for chunk in raw_chunks if not is_low_signal_chunk(chunk["content"])]
filtered_chunks = len(raw_chunks) - len(chunks)

source_summary = pd.DataFrame({
    "documents": [len(docs)],
    "raw_chunks": [len(raw_chunks)],
    "usable_chunks": [len(chunks)],
    "filtered_low_signal_chunks": [filtered_chunks],
    "avg_chunk_chars": [round(sum(len(c["content"]) for c in chunks) / max(len(chunks), 1))],
})
show_table(
    "Geladene Quellen und Chunks",
    "Diese Tabelle prüft, ob die gewünschte Quelle gefunden wurde, wie viele Chunks entstanden sind und wie viele offensichtliche Literatur-/Metadaten-Chunks ausgefiltert wurden.",
    source_summary,
)


## 2. Testgraph leeren

Dieser Abschnitt stellt sicher, dass der folgende Lauf isoliert ist und keine alten Testdaten aus einem vorherigen Szenario in die Ergebnisse hineinspielen.

**Was der Code macht:** Er verbindet sich mit Neo4j und leert die konfigurierte Testdatenbank, sofern `RESET_GRAPH_ON_IMPORT` aktiv ist.

**Was du danach sehen solltest:** Eine kurze Bestätigung mit Datenbankname und `reset_graph_on_import`. Für die isolierten Demo-Läufe sollte dieser Wert `true` sein.

In [ ]:
# Neo4j-Verbindung vorbereiten und den Testgraphen optional vor jedem Szenario leeren.
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USER = os.getenv("NEO4J_USER") or os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
RESET_GRAPH_ON_IMPORT = os.getenv("RESET_GRAPH_ON_IMPORT", "true").lower() in {"1", "true", "yes", "ja"}

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def neo4j_session():
    if NEO4J_DATABASE:
        return driver.session(database=NEO4J_DATABASE)
    return driver.session()

with neo4j_session() as session:
    # Für den Spike ist ein sauberer Neustart hilfreich: jedes Notebook baut sein Szenario neu auf.
    if RESET_GRAPH_ON_IMPORT:
        session.run("MATCH (n) DETACH DELETE n")

show_json(
    "Testgraph zurückgesetzt",
    "Diese Ausgabe zeigt, welche Neo4j-Datenbank verwendet wurde und ob der Graph vor diesem Szenario geleert wurde.",
    {"database": NEO4J_DATABASE or "<default>", "reset_graph_on_import": RESET_GRAPH_ON_IMPORT},
)


## 3. Embeddings erzeugen

Dieser Abschnitt übersetzt jeden Text-Chunk in einen numerischen Vektor. Diese Vektoren sind die Grundlage für semantische Suche.

**Was der Code macht:** Er schickt die Chunk-Texte an das konfigurierte Embedding-Modell und hängt den resultierenden Vektor an jeden Chunk.

**Was du danach sehen solltest:** Eine kurze Zusammenfassung mit Anzahl der eingebetteten Chunks und Vektordimension. Beim Default-Embedding sollte die Dimension `1536` sein.

In [ ]:
# Embeddings: Jeder Chunk wird zu einem Vektor, damit Neo4j später semantisch suchen kann.
def create_llm_client() -> OpenAI:
    if LLM_PROVIDER == "github":
        api_key = (
            os.getenv("GITHUB_MODELS_API_KEY")
            or os.getenv("GITHUB_TOKEN")
            or os.getenv("LLM_API_KEY")
        )
        if not api_key:
            raise RuntimeError(
                "Für LLM_PROVIDER=github muss GITHUB_MODELS_API_KEY, GITHUB_TOKEN oder LLM_API_KEY gesetzt sein."
            )
        return OpenAI(
            api_key=api_key,
            base_url=GITHUB_MODELS_BASE_URL,
            default_headers={
                "Accept": "application/vnd.github+json",
                "X-GitHub-Api-Version": GITHUB_MODELS_API_VERSION,
            },
        )

    api_key = os.getenv("OPENAI_API_KEY") or os.getenv("LLM_API_KEY")
    if not api_key:
        raise RuntimeError("Für LLM_PROVIDER=openai muss OPENAI_API_KEY oder LLM_API_KEY gesetzt sein.")
    return OpenAI(api_key=api_key)


client = create_llm_client()


# Batching reduziert API-Overhead und hält den Ablauf auch bei vielen Chunks stabiler.
def embed_batch(texts: list[str], batch_size: int = 96) -> list[list[float]]:
    vectors: list[list[float]] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch,
            encoding_format="float",
        )
        vectors.extend(item.embedding for item in response.data)
    return vectors

vectors = embed_batch([c["content"] for c in chunks])
for chunk, vector in zip(chunks, vectors):
    chunk["embedding"] = vector

show_json(
    "Embeddings erzeugt",
    "Diese Werte zeigen, wie viele Chunks eingebettet wurden und welche Vektordimension das konfigurierte Embedding-Modell geliefert hat.",
    {
        "provider": LLM_PROVIDER,
        "model": EMBEDDING_MODEL,
        "chunks": len(chunks),
        "embedding_dimensions": len(chunks[0]["embedding"]),
    },
)


## 4. Ontologie laden und verstehen

Dieser Abschnitt macht die Kontextkarte sichtbar, bevor Daten nach Neo4j geschrieben werden.

**Was der Code macht:** Er lädt die passende JSON-Ontologie aus `ontologies/`, prüft Klassen, Entitäten und Relationen, extrahiert einfache Term- und Entity-Treffer aus den Chunks und zeigt dir Tabellen zur Ontologie.

**Was du danach sehen solltest:** Eine kurze Ontologie-Zusammenfassung, eine Klassentabelle, eine Entitätentabelle und eine Relationentabelle. Das ist der Moment, in dem du prüfen kannst: Ergibt diese Kontextkarte fachlich Sinn?


In [ ]:
# Schema und Ontologie: Constraints sichern eindeutige Knoten, der Vector Index macht Ähnlichkeitssuche möglich.
CREATE_SCHEMA_STATEMENTS = [
    "CREATE CONSTRAINT doc_id IF NOT EXISTS FOR (d:Document) REQUIRE d.id IS UNIQUE",
    "CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT term_name IF NOT EXISTS FOR (t:Term) REQUIRE t.name IS UNIQUE",
    "CREATE CONSTRAINT concept_id IF NOT EXISTS FOR (c:Concept) REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE",
    "CREATE CONSTRAINT ontology_id IF NOT EXISTS FOR (o:Ontology) REQUIRE o.id IS UNIQUE",
    "CREATE CONSTRAINT ontology_class_id IF NOT EXISTS FOR (c:OntologyClass) REQUIRE c.id IS UNIQUE",
    """
    CREATE VECTOR INDEX chunk_vec_idx IF NOT EXISTS
    FOR (c:Chunk) ON c.embedding
    OPTIONS { indexConfig: {
      `vector.dimensions`: $dim,
      `vector.similarity_function`: 'cosine',
      `vector.hnsw.m`: 16,
      `vector.hnsw.ef_construction`: 128,
      `vector.quantization.enabled`: false
    }}
    """,
]

ONTOLOGY_FILE_BY_SCENARIO = {
    "klima_kommunale_klimaanpassung": "ontologies/klima_kommunale_klimaanpassung.json",
    "bsi_it_grundschutz": "ontologies/bsi_it_grundschutz.json",
    "sozialbericht_2024": "ontologies/sozialbericht_2024.json",
}

ONTOLOGY_PATH = ROOT / ONTOLOGY_FILE_BY_SCENARIO.get(SCENARIO_ID, f"ontologies/{SCENARIO_ID}.json")


# Die Ontologie ist die kuratierte fachliche Kontextkarte für das Szenario.
def _read_ontology(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Ontologie-Datei nicht gefunden: {path}")
    payload = json.loads(path.read_text(encoding="utf-8"))
    payload.setdefault("ontology_id", path.stem)
    payload.setdefault("version", "0.0.0")
    payload.setdefault("title", path.stem)
    payload.setdefault("entities", [])
    payload.setdefault("classes", [])
    payload.setdefault("relations", [])
    return payload


def _safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9äöüß]+", "_", value.lower().strip()).strip("_")


# IDs werden mit der Ontologie qualifiziert, damit Szenarien später nicht kollidieren.
def _qualified_node_id(ontology_id: str, local_id: str) -> str:
    return f"{ontology_id}:{_safe_slug(local_id)}"


# Aus JSON werden flache Importlisten, die Neo4j per UNWIND effizient schreiben kann.
def load_ontology_model(ontology: dict, scenario_id: str) -> tuple[dict, list[dict], list[dict], list[dict]]:
    ontology_id = str(ontology["ontology_id"])

    class_rows = []
    class_by_id: dict[str, str] = {}
    for item in ontology.get("classes", []):
        raw_id = item.get("id") or item.get("name")
        class_id = _qualified_node_id(ontology_id, str(raw_id))
        class_by_id[str(raw_id)] = class_id
        class_rows.append(
            {
                "id": class_id,
                "ontology_id": ontology_id,
                "name": item.get("name", raw_id),
                "description": item.get("description", ""),
            }
        )

    entity_rows = []
    for item in ontology.get("entities", []):
        raw_id = item.get("id") or item.get("name")
        class_ref = item.get("class_id")
        class_id = class_by_id.get(str(class_ref)) if class_ref is not None else None
        if class_id is None and class_ref is not None:
            class_id = _qualified_node_id(ontology_id, str(class_ref))
            if not any(row["id"] == class_id for row in class_rows):
                class_rows.append(
                    {
                        "id": class_id,
                        "ontology_id": ontology_id,
                        "name": str(class_ref),
                        "description": "",
                    }
                )

        entity_rows.append(
            {
                "id": _qualified_node_id(ontology_id, str(raw_id)),
                "ontology_id": ontology_id,
                "scenario": scenario_id,
                "name": item.get("name", raw_id),
                "description": item.get("description", ""),
                "class_id": class_id,
                "aliases": item.get("aliases", []),
            }
        )

    relation_rows = []
    for item in ontology.get("relations", []):
        source = item.get("source_id")
        target = item.get("target_id")
        if not source or not target:
            continue
        relation_rows.append(
            {
                "id": _qualified_node_id(ontology_id, str(item.get("id") or f"{source}-to-{target}")),
                "ontology_id": ontology_id,
                "label": item.get("label", "bezieht_sich_auf"),
                "source_id": _qualified_node_id(ontology_id, str(source)),
                "target_id": _qualified_node_id(ontology_id, str(target)),
                "description": item.get("description", ""),
                "bidirectional": bool(item.get("bidirectional", False)),
            }
        )

    return ontology, class_rows, entity_rows, relation_rows


def extract_entities_from_text(text: str, entities: list[dict]) -> list[dict]:
    text_lower = text.lower()
    found: list[dict] = []
    seen: set[str] = set()

    for row in entities:
        labels = [row["name"]] + list(row.get("aliases", []))
        matches: list[str] = []
        for label in labels:
            candidate = str(label).lower().strip()
            if candidate and candidate in text_lower:
                matches.append(candidate)

        if matches:
            if row["id"] in seen:
                continue
            seen.add(row["id"])
            found.append(
                {
                    "id": row["id"],
                    "name": row["name"],
                    "class_id": row.get("class_id"),
                    "matched_aliases": sorted(set(matches)),
                }
            )

    return found


def neo4j_session():
    if NEO4J_DATABASE:
        return driver.session(database=NEO4J_DATABASE)
    return driver.session()


def simple_terms(text: str, limit: int = 6) -> list[str]:
    # Harte Heuristik für deutsche Fachbegriffe: Wir nehmen primär großgeschriebene Nomen aus dem Originaltext.
    # Das ist absichtlich konservativ. Lieber weniger Terms als Hilfsverben wie "werden" im Graphen.
    stop = {
        "aber", "alle", "allem", "allen", "aller", "alles", "also", "auch", "auf", "aus", "beim", "bereits",
        "dabei", "dadurch", "dafür", "damit", "danach", "dann", "dass", "dazu", "dem", "den", "der", "des",
        "diese", "diesem", "diesen", "dieser", "dieses", "doch", "durch", "eine", "einem", "einen", "einer", "eines",
        "erst", "etwa", "für", "ganz", "gegen", "haben", "hier", "ihre", "ihrem", "ihren", "ihrer", "ihres", "immer",
        "kann", "können", "mehr", "mit", "nach", "nicht", "noch", "oder", "ohne", "sein", "seine", "seinem", "seinen",
        "seiner", "sich", "sind", "sowie", "über", "und", "uns", "unter", "vom", "von", "vor", "wäre", "weil", "wenn",
        "werden", "wird", "wurden", "wurde", "zur", "zwar", "zwischen", "this", "that", "with", "from",
    }
    generic = {
        "abbildung", "beispiel", "beispiele", "bereich", "daten", "ergebnis", "fall", "form", "grund", "hinweis",
        "kapitel", "kontext", "modell", "personen", "prozess", "punkt", "seite", "stellen", "system", "tabelle", "teil",
        "text", "thema", "unterschied", "weise", "wert", "werte", "ziel",
    }
    noun_candidates = re.findall(
        r"(?<![A-Za-zÄÖÜäöüß-])[A-ZÄÖÜ][A-Za-zÄÖÜäöüß]{4,}(?:-[A-ZÄÖÜ]?[A-Za-zÄÖÜäöüß]{3,})*",
        text,
    )
    normalized = [re.sub(r"^[^A-Za-zÄÖÜäöüß]+|[^A-Za-zÄÖÜäöüß]+$", "", candidate).lower() for candidate in noun_candidates]
    terms = [
        term
        for term in normalized
        if term
        and term not in stop
        and term not in generic
        and len(term) >= 5
        and not term.endswith(("ende", "enden", "ender", "endes"))
    ]
    if not terms:
        return []
    return pd.Series(terms).value_counts().head(limit).index.tolist()


ONTOLOGY = _read_ontology(ONTOLOGY_PATH)
ONTOLOGY, CLASS_ROWS, ENTITY_ROWS, ONTOLOGY_RELATION_ROWS = load_ontology_model(ONTOLOGY, SCENARIO_ID)

term_rows: list[dict] = []
mention_rows: list[dict] = []
matched_entity_rows_by_id: dict[str, dict] = {}
has_entity_rows: list[dict] = []
cooccurrence_rows = set()

for chunk in chunks:
    for term in simple_terms(chunk["content"]):
        term_rows.append({"name": term})
        mention_rows.append({"chunk_id": chunk["chunk_id"], "term": term})

    extracted_entities = extract_entities_from_text(chunk["content"], ENTITY_ROWS)
    for entity in extracted_entities:
        matched_entity_rows_by_id.setdefault(entity["id"], entity)
        has_entity_rows.append(
            {
                "chunk_id": chunk["chunk_id"],
                "entity_id": entity["id"],
                "matched_aliases": entity["matched_aliases"],
            }
        )

    entity_ids = [row["id"] for row in extracted_entities]
    for left_index, left in enumerate(entity_ids):
        for right in entity_ids[left_index + 1 :]:
            pair = tuple(sorted([left, right]))
            cooccurrence_rows.add(pair)

matched_entity_rows = list(matched_entity_rows_by_id.values())
cooccurrence_payload = [{"left_id": left, "right_id": right} for left, right in sorted(cooccurrence_rows)]
class_name_by_local_id = {item.get("id"): item.get("name", item.get("id")) for item in ONTOLOGY.get("classes", [])}
entity_name_by_local_id = {item.get("id"): item.get("name", item.get("id")) for item in ONTOLOGY.get("entities", [])}

ontology_class_table = pd.DataFrame([
    {
        "klasse": item.get("name", item.get("id")),
        "id": item.get("id"),
        "beschreibung": item.get("description", ""),
    }
    for item in ONTOLOGY.get("classes", [])
])

ontology_entity_table = pd.DataFrame([
    {
        "entität": item.get("name", item.get("id")),
        "klasse": class_name_by_local_id.get(item.get("class_id"), item.get("class_id")),
        "aliases": ", ".join(item.get("aliases", [])[:5]),
        "beschreibung": item.get("description", ""),
    }
    for item in ONTOLOGY.get("entities", [])
])

ontology_relation_table = pd.DataFrame([
    {
        "von": entity_name_by_local_id.get(item.get("source_id"), item.get("source_id")),
        "relation": item.get("label", "bezieht sich auf"),
        "nach": entity_name_by_local_id.get(item.get("target_id"), item.get("target_id")),
        "beschreibung": item.get("description", ""),
    }
    for item in ONTOLOGY.get("relations", [])
])

show_json(
    "Ontologie geladen",
    "Diese Zahlen beschreiben die Kontextkarte, die gleich vollständig in Neo4j landet. `matched_entities` zeigt, wie viele Ontologie-Entitäten in den aktuellen Chunks über Aliase gefunden wurden.",
    {
        "ontology_path": str(ONTOLOGY_PATH),
        "ontology_id": ONTOLOGY.get("ontology_id"),
        "version": ONTOLOGY.get("version"),
        "title": ONTOLOGY.get("title"),
        "classes": len(ONTOLOGY.get("classes", [])),
        "entities": len(ONTOLOGY.get("entities", [])),
        "relations": len(ONTOLOGY.get("relations", [])),
        "matched_entities_in_chunks": len(matched_entity_rows),
        "chunk_entity_links": len(has_entity_rows),
    },
)

if ONTOLOGY.get("reading_guide"):
    display(Markdown(f"### Leselogik der Ontologie\n\n{ONTOLOGY['reading_guide']}"))

show_table(
    "Klassen: Welche Arten von Knoten gibt es?",
    "Klassen sind keine zusätzlichen Themen, sondern Typen. Sie beantworten: Welche Art von Ding ist eine Entität?",
    ontology_class_table,
)
show_table(
    "Entitäten: Welche fachlichen Knoten liegen im Graph?",
    "Entitäten sind die konkreten fachlichen Begriffe der Kontextkarte. Über Aliase werden sie später mit Chunks verbunden.",
    ontology_entity_table,
)
show_table(
    "Relationen: Wie hängen die Entitäten zusammen?",
    "Relationen sind kuratierte Kanten der Ontologie. Sie sind die eigentliche Kontextkarte, die du später in Neo4j anschauen kannst.",
    ontology_relation_table,
)


## 5. Neo4j Schema und Import

Dieser Abschnitt schreibt Dokumentgraph und Ontologiegraph in Neo4j.

**Was der Code macht:** Er erstellt Constraints und den Vector Index, importiert die vollständige Ontologie, legt Dokumente und Chunks an und verbindet Chunks über `HAS_CONCEPT` mit gefundenen Ontologie-Entitäten.

**Was du danach sehen solltest:** Eine Import-Zusammenfassung. Wichtig sind `ontology_entities`, `matched_entities`, `entity_links`, `chunks` und `reset_graph_on_import`.


In [ ]:
# Import in Neo4j: erst Schema, dann Ontologie, Dokumente, Chunks und Verknüpfungen.
with neo4j_session() as session:
    for statement in CREATE_SCHEMA_STATEMENTS:
        session.run(statement, dim=EMBEDDING_DIM)

    session.run(
        """
        MERGE (o:Ontology {id:$id})
        SET o.title = $title,
            o.version = $version,
            o.scenario = $scenario,
            o.domain = $domain,
            o.updated_at = datetime()
        """,
        id=ONTOLOGY["ontology_id"],
        title=ONTOLOGY.get("title", ""),
        version=ONTOLOGY.get("version", ""),
        scenario=SCENARIO_ID,
        domain=ONTOLOGY.get("domain", SCENARIO_ID),
    )

    if CLASS_ROWS:
        session.run(
            """
            UNWIND $rows AS r
            MERGE (class:OntologyClass {id:r.id})
            SET class.ontology_id = r.ontology_id,
                class.name = r.name,
                class.description = r.description
            """,
            rows=CLASS_ROWS,
        )
        session.run(
            """
            MATCH (o:Ontology {id:$ontology_id})
            UNWIND $rows AS r
            MATCH (class:OntologyClass {id:r.id})
            MERGE (o)-[:HAS_CLASS]->(class)
            """,
            ontology_id=ONTOLOGY["ontology_id"],
            rows=CLASS_ROWS,
        )

    # Entitäten sind zugleich Concept-Knoten, damit Retrieval und Ontologie denselben Fachknoten nutzen.
    if ENTITY_ROWS:
        session.run(
            """
            UNWIND $rows AS r
            MERGE (entity:Entity:Concept {id:r.id})
            SET entity.ontology_id = r.ontology_id,
                entity.scenario = r.scenario,
                entity.name = r.name,
                entity.description = r.description,
                entity.aliases = r.aliases
            """,
            rows=ENTITY_ROWS,
        )
        class_links = [r for r in ENTITY_ROWS if r.get("class_id")]
        if class_links:
            session.run(
                """
                UNWIND $rows AS r
                MATCH (entity:Entity {id:r.id})
                MATCH (class:OntologyClass {id:r.class_id})
                MERGE (entity)-[:INSTANCE_OF]->(class)
                """,
                rows=class_links,
            )

    if ONTOLOGY_RELATION_ROWS:
        session.run(
            """
            UNWIND $rows AS r
            MATCH (left:Concept {id:r.source_id}), (right:Concept {id:r.target_id})
            MERGE (left)-[rel:ONTOLOGY_RELATION]->(right)
            SET rel.id = r.id,
                rel.label = r.label,
                rel.description = r.description,
                rel.ontology_id = r.ontology_id,
                rel.bidirectional = r.bidirectional,
                rel.scenario = $scenario
            """,
            scenario=SCENARIO_ID,
            rows=ONTOLOGY_RELATION_ROWS,
        )

    session.run(
        "UNWIND $rows AS r MERGE (d:Document {id:r.id}) SET d.name=r.name, d.path=r.path, d.scenario = $scenario",
        scenario=SCENARIO_ID,
        rows=[{"id": d["doc_id"], "name": d["doc_id"], "path": d["path"]} for d in docs],
    )
    session.run(
        """
        UNWIND $rows AS r
        MERGE (c:Chunk {id:r.id})
        SET c.text = r.text,
            c.embedding = r.embedding,
            c.doc_id = r.doc_id,
            c.scenario = r.scenario
        WITH c, r
        MATCH (d:Document {id:r.doc_id})
        MERGE (d)-[:HAS_CHUNK]->(c)
        """,
        rows=[{"id": c["chunk_id"], "doc_id": c["doc_id"], "text": c["content"], "embedding": c["embedding"], "scenario": SCENARIO_ID} for c in chunks],
    )
    if term_rows:
        session.run("UNWIND $rows AS r MERGE (:Term {name:r.name})", rows=term_rows)
        session.run(
            """
            UNWIND $rows AS r
            MATCH (c:Chunk {id:r.chunk_id}), (t:Term {name:r.term})
            MERGE (c)-[:MENTIONS]->(t)
            """,
            rows=mention_rows,
        )
    if has_entity_rows:
        session.run(
            """
            UNWIND $rows AS r
            MATCH (chunk:Chunk {id:r.chunk_id}), (concept:Concept {id:r.entity_id})
            MERGE (chunk)-[rel:HAS_CONCEPT]->(concept)
            SET rel.matched_aliases = r.matched_aliases
            """,
            rows=has_entity_rows,
        )
    if cooccurrence_payload:
        session.run(
            """
            UNWIND $rows AS r
            MATCH (left:Concept {id:r.left_id}), (right:Concept {id:r.right_id})
            MERGE (left)-[rel:CO_OCCURS_WITH]-(right)
            SET rel.label = "tritt gemeinsam auf mit"
            """,
            rows=cooccurrence_payload,
        )

show_json(
    "Graph importiert",
    "Diese Zusammenfassung zeigt den Dokumentengraphen und den Ontologiegraphen.",
    {
        "documents": len(docs),
        "chunks": len(chunks),
        "terms": len({r["name"] for r in term_rows}),
        "matched_entities": len(matched_entity_rows),
        "entity_links": len(has_entity_rows),
        "cooccurrence_edges": len(cooccurrence_payload),
        "ontology_relations": len(ONTOLOGY_RELATION_ROWS),
        "ontology_id": ONTOLOGY["ontology_id"],
        "ontology_version": ONTOLOGY.get("version", ""),
        "ontology_classes": len(CLASS_ROWS),
        "database": NEO4J_DATABASE or "<default>",
        "reset_graph_on_import": RESET_GRAPH_ON_IMPORT,
    },
)


## 6. Graph in Neo4j prüfen

Dieser Abschnitt ist der kleine Reality-Check nach dem Import.

**Was der Code macht:** Er fragt Neo4j direkt ab und zeigt, welche Klassen, Entitäten und Relationen wirklich in der Datenbank liegen. Zusätzlich bekommst du Cypher-Abfragen, die du im Neo4j Browser/Aura Explorer ausführen kannst.

**Was du danach sehen solltest:** Zählwerte und Beispiel-Relationen. Wenn hier Entitäten und Relationen auftauchen, ist die Kontextkarte wirklich im Graph angekommen.


In [ ]:

# Kapitel 6 enthält zwei Ebenen: Live-Checks fürs Notebook und Cypher zum Kopieren nach Neo4j.
GRAPH_COUNTS_CYPHER = """
MATCH (o:Ontology {id:$ontology_id})
OPTIONAL MATCH (o)-[:HAS_CLASS]->(class:OntologyClass)
WITH o, count(DISTINCT class) AS classes
OPTIONAL MATCH (entity:Entity {ontology_id:$ontology_id})
WITH o, classes, count(DISTINCT entity) AS entities
OPTIONAL MATCH (:Entity {ontology_id:$ontology_id})-[rel:ONTOLOGY_RELATION]->(:Entity {ontology_id:$ontology_id})
RETURN o.id AS ontology_id, o.title AS title, classes, entities, count(rel) AS ontology_relations
"""

GRAPH_RELATIONS_CYPHER = """
MATCH (left:Entity {ontology_id:$ontology_id})-[rel:ONTOLOGY_RELATION]->(right:Entity {ontology_id:$ontology_id})
RETURN left.name AS von, rel.label AS relation, right.name AS nach
ORDER BY von, relation, nach
LIMIT 20
"""

GRAPH_CLASSES_CYPHER = """
MATCH (class:OntologyClass)<-[:INSTANCE_OF]-(entity:Entity {ontology_id:$ontology_id})
RETURN class.name AS klasse, collect(entity.name)[0..12] AS entitaeten
ORDER BY klasse
"""

try:
    with neo4j_session() as session:
        graph_counts = session.run(GRAPH_COUNTS_CYPHER, ontology_id=ONTOLOGY["ontology_id"]).single().data()
        graph_relations = session.run(GRAPH_RELATIONS_CYPHER, ontology_id=ONTOLOGY["ontology_id"]).data()
        graph_classes = session.run(GRAPH_CLASSES_CYPHER, ontology_id=ONTOLOGY["ontology_id"]).data()

    show_json(
        "Neo4j-Graph geprüft",
        "Diese Zahlen kommen direkt aus Neo4j und bestätigen, dass die Ontologie als eigener Graph importiert wurde.",
        graph_counts,
    )
    show_table(
        "Beispiel-Relationen aus Neo4j",
        "Diese Kanten sind die kuratierte Kontextkarte. Genau diese Struktur solltest du dir später grafisch in Neo4j ansehen.",
        pd.DataFrame(graph_relations),
    )
    show_table(
        "Klassen mit Entitäten aus Neo4j",
        "Diese Tabelle zeigt noch einmal den Unterschied: Klassen sind Typen, Entitäten sind die konkreten fachlichen Knoten.",
        pd.DataFrame(graph_classes),
    )
except Exception as exc:  # noqa: BLE001
    display(Markdown(
        "### Neo4j ist gerade nicht erreichbar\n\n"
        "Die Cypher-Beispiele oben sind trotzdem nutzbar. Der Live-Check konnte keine Verbindung zu Neo4j aufbauen. "
        "Typische Ursachen sind: Aura-Instanz schläft, Netzwerk/VPN blockiert, `NEO4J_URI` passt nicht oder die Credentials sind abgelaufen.\n\n"
        f"Technische Meldung: `{type(exc).__name__}: {exc}`"
    ))


## 6b. Cypher-Queries zum Kopieren

Diese Queries kannst du direkt in Neo4j Browser oder Aura Explorer einfügen. Kopiere nur den Inhalt des jeweiligen `cypher`-Blocks, nicht die Überschrift.

Die Tabellen-Queries geben Spalten zurück und sind gut zum Prüfen. Die Graph-Queries geben Knoten und Beziehungen zurück; dort kannst du in Neo4j auf die Graph-Ansicht wechseln.

Neo4j zeigt im Graph-Canvas häufig den technischen Relationstyp, zum Beispiel `CO_OCCURS_WITH`. Das lesbare deutsche Kantenlabel steht als Relationship-Property `label`, etwa `tritt gemeinsam auf mit`. In den Notebook-Ausgaben wird dieses Label automatisch verwendet.

### Tabellen-Queries

```cypher
// 1. Überblick: Wie groß ist die importierte Ontologie?
WITH "sozialbericht_2024" AS ontology_id
MATCH (o:Ontology {id: ontology_id})
OPTIONAL MATCH (o)-[:HAS_CLASS]->(class:OntologyClass)
WITH o, ontology_id, count(DISTINCT class) AS klassen
OPTIONAL MATCH (entity:Entity {ontology_id: ontology_id})
WITH o, ontology_id, klassen, count(DISTINCT entity) AS entitaeten
OPTIONAL MATCH (:Entity {ontology_id: ontology_id})-[rel:ONTOLOGY_RELATION]->(:Entity {ontology_id: ontology_id})
RETURN o.title AS ontologie, klassen, entitaeten, count(rel) AS relationen;
```

```cypher
// 2. Klassen und ihre konkreten Entitäten.
WITH "sozialbericht_2024" AS ontology_id
MATCH (class:OntologyClass)<-[:INSTANCE_OF]-(entity:Entity {ontology_id: ontology_id})
RETURN class.name AS klasse, collect(entity.name)[0..20] AS entitaeten
ORDER BY klasse;
```

```cypher
// 3. Fachliche Beziehungen der Ontologie als Tabelle.
WITH "sozialbericht_2024" AS ontology_id
MATCH (left:Entity {ontology_id: ontology_id})-[rel:ONTOLOGY_RELATION]->(right:Entity {ontology_id: ontology_id})
RETURN left.name AS von, rel.label AS beziehung, right.name AS nach
ORDER BY von, beziehung, nach
LIMIT 50;
```

### Graph-Queries

```cypher
// 4. Ontologie-Graph: Klassen, Entitäten und fachliche Beziehungen.
WITH "sozialbericht_2024" AS ontology_id
MATCH (o:Ontology {id: ontology_id})
OPTIONAL MATCH (o)-[has_class:HAS_CLASS]->(class:OntologyClass)<-[instance_of:INSTANCE_OF]-(entity:Entity {ontology_id: ontology_id})
OPTIONAL MATCH (entity)-[r:ONTOLOGY_RELATION]->(neighbor:Entity {ontology_id: ontology_id})
RETURN o, has_class, class, instance_of, entity, r, neighbor
LIMIT 300;
```

```cypher
// 5. Dokumenten-Graph: Dokument, Chunks, Ontologie-Treffer und einfache Terms.
MATCH (d:Document)-[has_chunk:HAS_CHUNK]->(chunk:Chunk)
OPTIONAL MATCH (chunk)-[has_concept:HAS_CONCEPT]->(concept:Concept {ontology_id: "sozialbericht_2024"})
OPTIONAL MATCH (chunk)-[mentions:MENTIONS]->(term:Term)
RETURN d, has_chunk, chunk, has_concept, concept, mentions, term
LIMIT 250;
```

```cypher
// 6. Kombinationsgraph: Dokument-Chunks plus Ontologie-Nachbarschaft.
MATCH (d:Document)-[has_chunk:HAS_CHUNK]->(chunk:Chunk)
OPTIONAL MATCH (chunk)-[has_concept:HAS_CONCEPT]->(concept:Concept {ontology_id: "sozialbericht_2024"})
OPTIONAL MATCH (concept)-[instance_of:INSTANCE_OF]->(class:OntologyClass)
OPTIONAL MATCH (concept)-[r:ONTOLOGY_RELATION|CO_OCCURS_WITH]-(related:Concept {ontology_id: "sozialbericht_2024"})
OPTIONAL MATCH (chunk)-[mentions:MENTIONS]->(term:Term)
RETURN d AS dokument, has_chunk, chunk, has_concept, concept, instance_of, class, related, r, mentions, term
LIMIT 300;
```


## 7. Vector Search plus Ontologie-Kontext

Dieser Abschnitt stellt die eigentliche Demo-Frage und sucht passende Beleg-Chunks im Graphen.

**Was der Code macht:** Er erzeugt ein Embedding für die Frage, fragt den Neo4j Vector Index ab und sammelt zusätzlich Ontologie-Entitäten, benachbarte Ontologie-Knoten, einfache Terms und Nachbar-Chunks ein.

**Was du danach sehen solltest:** Eine Tabelle der Top-Treffer mit `chunk_id`, `score`, `ontology_entities`, `ontology_neighbors` und `terms`. Gute Treffer sollten fachlich zur Frage passen und sinnvolle Ontologie-Knoten zeigen.


In [ ]:
# Die Frage wird ebenfalls eingebettet; Neo4j sucht danach die ähnlichsten Chunk-Vektoren.
question_vector = embed_batch([QUESTION])[0]

# Vector Search plus 1-Hop-Kontext: Treffer werden mit Terms und Ontologie-Nachbarn angereichert.
SEARCH_CYPHER = '''
CALL db.index.vector.queryNodes('chunk_vec_idx', $k, $query_vector)
YIELD node, score
WITH node, score
MATCH (node)<-[:HAS_CHUNK]-(doc:Document)
CALL (node) {
  OPTIONAL MATCH (node)-[:MENTIONS]->(term:Term)<-[:MENTIONS]-(neighbor:Chunk)
  RETURN collect(DISTINCT term.name)[0..8] AS terms,
         collect(DISTINCT neighbor.id)[0..8] AS neighbors
}
CALL (node) {
  OPTIONAL MATCH (node)-[:HAS_CONCEPT]->(concept:Concept)
  OPTIONAL MATCH (concept)-[:INSTANCE_OF]->(class:OntologyClass)
  RETURN collect(DISTINCT {id: concept.id, name: concept.name, class_name: class.name})[0..8] AS concepts
}
CALL (node) {
  OPTIONAL MATCH (node)-[:HAS_CONCEPT]->(concept:Concept)-[rel:ONTOLOGY_RELATION|CO_OCCURS_WITH]-(related:Concept)
  OPTIONAL MATCH (related)-[:INSTANCE_OF]->(related_class:OntologyClass)
  RETURN collect(DISTINCT {id: related.id, name: related.name, class_name: related_class.name, relation: coalesce(rel.label, CASE type(rel) WHEN "CO_OCCURS_WITH" THEN "tritt gemeinsam auf mit" ELSE type(rel) END)})[0..8] AS related_concepts
}
RETURN node.id AS chunk_id,
       doc.id AS doc_id,
       node.text AS text,
       score,
       terms,
       neighbors,
       [concept IN concepts WHERE concept.id IS NOT NULL] AS concepts,
       [concept IN related_concepts WHERE concept.id IS NOT NULL] AS related_concepts
ORDER BY score DESC
LIMIT $k
'''

with neo4j_session() as session:
    rows = session.run(SEARCH_CYPHER, k=TOP_K, query_vector=question_vector).data()


# Normalisierung macht Notebook-Ausgaben robuster, weil Neo4j-Werte je nach Query als Liste, String oder Dict ankommen können.
def _normalize_list_value(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [item.strip() for item in value.split(",") if item.strip()]
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    if isinstance(value, dict):
        items = value.get("items")
        if isinstance(items, (list, tuple, set)):
            return [str(item).strip() for item in items if str(item).strip()]
    return [str(value).strip()] if str(value).strip() else []


# Interne Neo4j-Relationstypen werden für Ausgaben in lesbare deutsche Kantenlabels übersetzt.
def readable_relation_label(label: str | None) -> str:
    relation = str(label or "").strip()
    label_map = {
        "CO_OCCURS_WITH": "tritt gemeinsam auf mit",
        "ONTOLOGY_RELATION": "ist fachlich verbunden mit",
        "HAS_CONCEPT": "erwähnt Entität",
        "HAS_CHUNK": "enthält Chunk",
        "MENTIONS": "erwähnt Begriff",
        "RELATED_1H": "führt zu Nachbar-Chunk",
        "GRAPH_EVIDENCE": "liefert Graph-Beleg",
    }
    return label_map.get(relation, relation)


# Für Tabellen werden Ontologie-Treffer in kurze lesbare Labels übersetzt.
def _format_concept_item(item):
    if not isinstance(item, dict):
        return str(item).strip()
    name = str(item.get("name", "")).strip()
    class_name = str(item.get("class_name") or "").strip()
    if name and class_name:
        return f"{name} ({class_name})"
    return name


def _format_related_concept_item(item):
    if not isinstance(item, dict):
        return str(item).strip()
    name = str(item.get("name", "")).strip()
    relation = readable_relation_label(item.get("relation"))
    if name and relation:
        return f"{relation}: {name}"
    return name


def _normalize_concepts(value):
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        if text.startswith("[") and text.endswith("]"):
            try:
                value = json.loads(text)
            except Exception:
                return [item.strip() for item in text.strip("[]").split(",") if item.strip()]
    if isinstance(value, list):
        out = [_format_concept_item(item) for item in value]
        return [item for item in out if item]
    if isinstance(value, (tuple, set)):
        return _normalize_concepts(list(value))
    if isinstance(value, dict):
        item = _format_concept_item(value)
        return [item] if item else []
    return [str(value)] if str(value).strip() else []


def _normalize_related_concepts(value):
    if value is None:
        return []
    if isinstance(value, list):
        out = [_format_related_concept_item(item) for item in value]
        return [item for item in out if item]
    if isinstance(value, (tuple, set)):
        return _normalize_related_concepts(list(value))
    if isinstance(value, dict):
        item = _format_related_concept_item(value)
        return [item] if item else []
    return _normalize_list_value(value)


search_results = pd.DataFrame([
    {
        "chunk_id": row.get("chunk_id", ""),
        "doc_id": row.get("doc_id", ""),
        "score": round(float(row.get("score", 0.0)), 4),
        "ontology_entities": ", ".join(_normalize_concepts(row.get("concepts", []))),
        "ontology_neighbors": ", ".join(_normalize_related_concepts(row.get("related_concepts", []))),
        "terms": _normalize_list_value(row.get("terms", [])),
    }
    for row in rows
])
show_table(
    "Top-Treffer der Vector Search",
    "Diese Tabelle ist der Kern von GraphRAG: semantische Nähe findet Chunks, der Graph liefert fachlichen Kontext dazu.",
    search_results,
)


## 8. Graph Expansion Light

Dieser Abschnitt ist der sichtbare GraphRAG-Schritt nach der Vector Search.

**Was der Code macht:** Er nimmt die Entitäten aus den direkten Chunk-Treffern, läuft bis zu zwei Hops über den Ontologiegraphen und sucht anschließend zusätzliche Chunks, die an diesen Pfad-Entitäten hängen.

**Was du danach sehen solltest:** Eine Tabelle mit Graphpfaden und eine Tabelle mit ergänzenden Graph-Belegen. Diese Zusatzbelege ersetzen die direkten Treffer nicht, sondern erklären und erweitern sie.

Interne Neo4j-Kantentypen wie `CO_OCCURS_WITH` bleiben im Datenmodell technische Kategorien. In Tabellen, Briefings und Antworten werden sie über `rel.label` lesbar angezeigt, zum Beispiel als `tritt gemeinsam auf mit`.


In [ ]:
# Graph Expansion Light: Erst jetzt wird aus Chunk-first-RAG sichtbar GraphRAG.
GRAPH_HOPS = 2
MAX_GRAPH_PATHS = 14
MAX_EXPANDED_CHUNKS = 6

seed_entity_ids = sorted({
    concept.get("id")
    for row in rows
    for concept in row.get("concepts", [])
    if concept.get("id")
})
direct_chunk_ids = sorted({row["chunk_id"] for row in rows})

GRAPH_PATHS_CYPHER = f"""
MATCH (seed:Concept)
WHERE seed.id IN $seed_entity_ids
MATCH path = (seed)-[:ONTOLOGY_RELATION|CO_OCCURS_WITH*1..{GRAPH_HOPS}]-(target:Concept)
WHERE target.id <> seed.id
  AND target.ontology_id = $ontology_id
WITH seed, target, path, relationships(path) AS rels, nodes(path) AS path_nodes
RETURN seed.id AS seed_id,
       seed.name AS seed_name,
       target.id AS target_id,
       target.name AS target_name,
       length(path) AS hops,
       [node IN path_nodes | {{id: node.id, name: node.name}}] AS path_nodes,
       [rel IN rels | coalesce(rel.label, CASE type(rel) WHEN "CO_OCCURS_WITH" THEN "tritt gemeinsam auf mit" ELSE type(rel) END)] AS rel_labels
ORDER BY hops, seed_name, target_name
LIMIT $max_paths
"""

EXPANDED_CHUNKS_CYPHER = """
MATCH (chunk:Chunk)-[:HAS_CONCEPT]->(via:Concept)
WHERE via.id IN $path_entity_ids
  AND NOT chunk.id IN $direct_chunk_ids
MATCH (doc:Document)-[:HAS_CHUNK]->(chunk)
CALL (chunk) {
  OPTIONAL MATCH (chunk)-[:MENTIONS]->(term:Term)
  RETURN collect(DISTINCT term.name)[0..8] AS terms
}
CALL (chunk) {
  OPTIONAL MATCH (chunk)-[:HAS_CONCEPT]->(concept:Concept)
  OPTIONAL MATCH (concept)-[:INSTANCE_OF]->(class:OntologyClass)
  RETURN collect(DISTINCT {id: concept.id, name: concept.name, class_name: class.name})[0..8] AS concepts
}
CALL (chunk) {
  OPTIONAL MATCH (chunk)-[:HAS_CONCEPT]->(:Concept)-[rel:ONTOLOGY_RELATION|CO_OCCURS_WITH]-(related:Concept)
  OPTIONAL MATCH (related)-[:INSTANCE_OF]->(related_class:OntologyClass)
  RETURN collect(DISTINCT {id: related.id, name: related.name, class_name: related_class.name, relation: coalesce(rel.label, CASE type(rel) WHEN "CO_OCCURS_WITH" THEN "tritt gemeinsam auf mit" ELSE type(rel) END)})[0..8] AS related_concepts
}
WITH chunk, doc, terms, concepts, related_concepts,
     collect(DISTINCT {id: via.id, name: via.name})[0..6] AS via_entities
RETURN chunk.id AS chunk_id,
       doc.id AS doc_id,
       chunk.text AS text,
       0.0 AS score,
       terms,
       [] AS neighbors,
       [concept IN concepts WHERE concept.id IS NOT NULL] AS concepts,
       [concept IN related_concepts WHERE concept.id IS NOT NULL] AS related_concepts,
       via_entities
ORDER BY size(via_entities) DESC, chunk.id
LIMIT $max_chunks
"""

if seed_entity_ids:
    with neo4j_session() as session:
        graph_paths = session.run(
            GRAPH_PATHS_CYPHER,
            seed_entity_ids=seed_entity_ids,
            ontology_id=ONTOLOGY["ontology_id"],
            max_paths=MAX_GRAPH_PATHS,
        ).data()

    path_entity_ids = sorted({
        node.get("id")
        for path in graph_paths
        for node in path.get("path_nodes", [])
        if node.get("id")
    })

    with neo4j_session() as session:
        expanded_rows = session.run(
            EXPANDED_CHUNKS_CYPHER,
            path_entity_ids=path_entity_ids,
            direct_chunk_ids=direct_chunk_ids,
            max_chunks=MAX_EXPANDED_CHUNKS,
        ).data()
else:
    graph_paths = []
    path_entity_ids = []
    expanded_rows = []


def format_graph_path(path: dict) -> str:
    nodes_in_path = [node.get("name", "") for node in path.get("path_nodes", []) if node.get("name")]
    rels = path.get("rel_labels", [])
    if not nodes_in_path:
        return ""
    parts = [nodes_in_path[0]]
    for rel, node_name in zip(rels, nodes_in_path[1:]):
        parts.append(f"--{readable_relation_label(rel)}-->")
        parts.append(node_name)
    return " ".join(parts)

graph_paths_table = pd.DataFrame([
    {
        "start_entität": path.get("seed_name", ""),
        "ziel_entität": path.get("target_name", ""),
        "hops": path.get("hops", 0),
        "pfad": format_graph_path(path),
    }
    for path in graph_paths
])
show_table(
    "Graphpfade aus den direkten Treffern",
    "Diese Pfade zeigen, wie die im Text gefundenen Entitäten fachlich im Ontologiegraphen weiterführen.",
    graph_paths_table,
)

expanded_results = pd.DataFrame([
    {
        "chunk_id": row.get("chunk_id", ""),
        "doc_id": row.get("doc_id", ""),
        "via_entities": ", ".join(entity.get("name", "") for entity in row.get("via_entities", [])),
        "ontology_entities": ", ".join(_normalize_concepts(row.get("concepts", []))),
        "terms": _normalize_list_value(row.get("terms", [])),
    }
    for row in expanded_rows
])
show_table(
    "Ergänzende Graph-Belege",
    "Diese Chunks wurden nicht direkt über Vector Search gefunden, sondern über Entitäten aus den Graphpfaden ergänzt.",
    expanded_results,
)


## 9. Provenienzgraph bauen

Dieser Abschnitt macht aus den Suchergebnissen einen kleinen Provenienzgraphen.

**Was der Code macht:** Er erzeugt Nodes und Edges für Dokumente, Treffer-Chunks, Ontologie-Entitäten, Term-Knoten und Nachbar-Chunks und speichert diese Struktur als `provenance.json`.

**Was du danach sehen solltest:** Eine Bestätigung, wo die JSON-Datei gespeichert wurde. Diese Datei ist die maschinenlesbare Grundlage für die spätere Graphansicht.


In [ ]:
# Aus den Suchergebnissen entsteht ein kleiner Provenienzgraph: Welche Quelle führte zu welchen Belegen?
nodes = []
edges = set()

# Jeder Treffer erzeugt Dokument-, Chunk-, Ontologie-, Term- und Nachbar-Knoten.
for row in rows:
    nodes.append({"id": row["doc_id"], "type": "Document", "label": row["doc_id"], "score": None})
    nodes.append({"id": row["chunk_id"], "type": "Chunk", "label": row["chunk_id"], "score": float(row["score"])})
    edges.add((row["doc_id"], row["chunk_id"], "HAS_CHUNK"))
    for concept in row.get("concepts", []):
        nodes.append({"id": concept["id"], "type": "Concept", "label": concept["name"], "score": None})
        edges.add((row["chunk_id"], concept["id"], "HAS_CONCEPT"))
    for related in row.get("related_concepts", []):
        nodes.append({"id": related["id"], "type": "Concept", "label": related["name"], "score": None})
    for concept in row.get("concepts", []):
        for related in row.get("related_concepts", []):
            if concept["id"] != related["id"]:
                edges.add((concept["id"], related["id"], readable_relation_label(related.get("relation"))))
    for term in row["terms"]:
        nodes.append({"id": f"term:{term}", "type": "Term", "label": term, "score": None})
        edges.add((row["chunk_id"], f"term:{term}", "MENTIONS"))
    for neighbor in row["neighbors"]:
        nodes.append({"id": neighbor, "type": "NeighborChunk", "label": neighbor, "score": max(float(row["score"]) - HOP_PENALTY, 0)})
        edges.add((row["chunk_id"], neighbor, "RELATED_1H"))

# Graphpfade kommen aus Abschnitt 8: Sie zeigen, warum der Graph zusätzliche Kontexte öffnen kann.
for path in globals().get("graph_paths", []):
    path_nodes = path.get("path_nodes", [])
    rel_labels = path.get("rel_labels", [])
    for node in path_nodes:
        if node.get("id") and node.get("name"):
            nodes.append({"id": node["id"], "type": "Concept", "label": node["name"], "score": None})
    for left, rel_label, right in zip(path_nodes, rel_labels, path_nodes[1:]):
        if left.get("id") and right.get("id"):
            edges.add((left["id"], right["id"], readable_relation_label(rel_label)))

# Ergänzende Graph-Belege sind Chunks, die über Pfad-Entitäten gefunden wurden, nicht direkt über Vector Search.
for row in globals().get("expanded_rows", []):
    nodes.append({"id": row["doc_id"], "type": "Document", "label": row["doc_id"], "score": None})
    nodes.append({"id": row["chunk_id"], "type": "GraphChunk", "label": row["chunk_id"], "score": None})
    edges.add((row["doc_id"], row["chunk_id"], "HAS_CHUNK"))
    for concept in row.get("concepts", []):
        nodes.append({"id": concept["id"], "type": "Concept", "label": concept["name"], "score": None})
        edges.add((row["chunk_id"], concept["id"], "HAS_CONCEPT"))
    for via in row.get("via_entities", []):
        nodes.append({"id": via["id"], "type": "Concept", "label": via["name"], "score": None})
        edges.add((via["id"], row["chunk_id"], "GRAPH_EVIDENCE"))

# Deduplizieren: derselbe Knoten kann in mehreren Treffern vorkommen, soll im Graph aber nur einmal existieren.
nodes = list({node["id"]: node for node in nodes}.values())
edges = [{"source": s, "target": t, "type": rel} for s, t, rel in sorted(edges)]

provenance = {
    "query": QUESTION,
    "sources": sorted({row["doc_id"] for row in rows}),
    "ontology": {
        "id": ONTOLOGY.get("ontology_id"),
        "title": ONTOLOGY.get("title"),
        "version": ONTOLOGY.get("version"),
    },
    "chunks": [
        {
            "id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "similarity": float(row["score"]),
            "ontology_entities": row.get("concepts", []),
            "ontology_neighbors": row.get("related_concepts", []),
            "terms": row["terms"],
            "neighbors": row["neighbors"],
            "text_preview": textwrap.shorten(row["text"].replace("\n", " "), width=320, placeholder="..."),
        }
        for row in rows
    ],
    "graph_paths": globals().get("graph_paths", []),
    "expanded_chunks": [
        {
            "id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "via_entities": row.get("via_entities", []),
            "ontology_entities": row.get("concepts", []),
            "text_preview": textwrap.shorten(row["text"].replace("\n", " "), width=320, placeholder="..."),
        }
        for row in globals().get("expanded_rows", [])
    ],
    "nodes": nodes,
    "edges": edges,
    "scoring": {"similarity": "cosine", "hop_penalty": HOP_PENALTY},
}

(OUTPUT_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8")
show_file(
    "Provenienz-JSON gespeichert",
    "Diese Datei enthält den maschinenlesbaren Provenienzgraphen mit Dokumenten, Chunks, Ontologie-Entitäten, Terms, Nachbarn und Textvorschauen.",
    OUTPUT_DIR / "provenance.json",
)


## 10. Graphansichten als HTML exportieren

Dieser Abschnitt erzeugt drei getrennte Graphansichten als eigenständige HTML-Dateien. Das Notebook zeigt die Graphen bewusst nicht mehr inline an, damit kein großes SVG den Jupyter-Hintergrund oder die Lesbarkeit des Notebooks beeinflusst.

**Was der Code macht:** Er baut aus den Retrieval-Ergebnissen einen Dokumentgraphen, aus der Ontologie einen Entity-Graphen und daraus eine kombinierte Ansicht. Alle drei Ansichten werden als HTML im szenariospezifischen Output-Ordner gespeichert.

**Was du danach sehen solltest:** Dateilinks zu `document_graph.html`, `entity_graph.html`, `combined_graph.html` und weiterhin `provenance.html` als kompatibler Alias auf die kombinierte Ansicht.


In [ ]:
# Graph-Export: Die Visualisierungen werden als HTML-Dateien geschrieben, nicht mehr inline ins Notebook gerendert.
from html import escape
from IPython.display import Markdown, display


def short_label(label: str, max_chars: int = 28) -> str:
    return label if len(label) <= max_chars else label[: max_chars - 1] + "…"


# Einfaches statisches Layout: Knotengruppen stehen in Spalten, damit die Demo schnell erfassbar bleibt.
def distribute(items: list[dict], x: int, y_top: int, y_bottom: int) -> list[dict]:
    if not items:
        return []
    if len(items) == 1:
        ys = [(y_top + y_bottom) / 2]
    else:
        step = (y_bottom - y_top) / (len(items) - 1)
        ys = [y_top + index * step for index in range(len(items))]
    return [{**item, "x": x, "y": round(y, 1), "label_short": short_label(item["label"])} for item, y in zip(items, ys)]


def unique_nodes(graph_nodes: list[dict]) -> list[dict]:
    return list({node["id"]: node for node in graph_nodes}.values())


def unique_edges(graph_edges: list[dict]) -> list[dict]:
    return list({(edge["source"], edge["target"], edge["type"]): edge for edge in graph_edges}.values())


# Dokumentgraph: Fokus auf Retrieval, Belege, Terms und 1-Hop-Nachbarn.
def build_document_graph(provenance_nodes: list[dict], provenance_edges: list[dict]) -> tuple[list[dict], list[dict]]:
    keep_types = {"Document", "Chunk", "GraphChunk", "NeighborChunk", "Term"}
    keep_edge_types = {"HAS_CHUNK", "MENTIONS", "RELATED_1H", "GRAPH_EVIDENCE"}
    graph_nodes = [node for node in provenance_nodes if node["type"] in keep_types]
    node_ids = {node["id"] for node in graph_nodes}
    graph_edges = [edge for edge in provenance_edges if edge["type"] in keep_edge_types and edge["source"] in node_ids and edge["target"] in node_ids]
    return unique_nodes(graph_nodes), unique_edges(graph_edges)


# Entity-Graph: die vollständige kuratierte Ontologie als Kontextkarte.
def build_entity_graph(ontology: dict) -> tuple[list[dict], list[dict]]:
    graph_nodes = [{"id": f"ontology:{ontology['ontology_id']}", "type": "Ontology", "label": ontology.get("title", ontology["ontology_id"]), "score": None}]
    graph_edges = []

    for ontology_class in ontology.get("classes", []):
        class_id = f"class:{ontology_class['id']}"
        graph_nodes.append({"id": class_id, "type": "OntologyClass", "label": ontology_class["name"], "score": None})
        graph_edges.append({"source": f"ontology:{ontology['ontology_id']}", "target": class_id, "type": "HAS_CLASS"})

    for entity in ontology.get("entities", []):
        graph_nodes.append({"id": entity["id"], "type": "Entity", "label": entity["name"], "score": None})
        graph_edges.append({"source": entity["id"], "target": f"class:{entity['class_id']}", "type": "INSTANCE_OF"})

    entity_ids = {entity["id"] for entity in ontology.get("entities", [])}
    for relation in ontology.get("relations", []):
        if relation["source_id"] in entity_ids and relation["target_id"] in entity_ids:
            graph_edges.append({"source": relation["source_id"], "target": relation["target_id"], "type": readable_relation_label(relation.get("label", "ONTOLOGY_RELATION"))})

    return unique_nodes(graph_nodes), unique_edges(graph_edges)


# Kombinierter Graph: verbindet Dokumenttreffer mit passenden Ontologie-Entitäten und Klassen.
def build_combined_graph(provenance_nodes: list[dict], provenance_edges: list[dict], ontology: dict) -> tuple[list[dict], list[dict]]:
    document_nodes, document_edges = build_document_graph(provenance_nodes, provenance_edges)
    concept_nodes = [node for node in provenance_nodes if node["type"] == "Concept"]
    concept_ids = {node["id"] for node in concept_nodes}
    entity_by_id = {entity["id"]: entity for entity in ontology.get("entities", [])}
    class_by_id = {ontology_class["id"]: ontology_class for ontology_class in ontology.get("classes", [])}
    class_ids = {entity_by_id[concept_id]["class_id"] for concept_id in concept_ids if concept_id in entity_by_id}

    graph_nodes = list(document_nodes)
    graph_nodes.extend({**node, "type": "Entity"} for node in concept_nodes)
    graph_nodes.append({"id": f"ontology:{ontology['ontology_id']}", "type": "Ontology", "label": ontology.get("title", ontology["ontology_id"]), "score": None})
    for class_id in sorted(class_ids):
        ontology_class = class_by_id.get(class_id)
        if ontology_class:
            graph_nodes.append({"id": f"class:{class_id}", "type": "OntologyClass", "label": ontology_class["name"], "score": None})

    graph_edges = list(document_edges)
    graph_edges.extend(edge for edge in provenance_edges if edge["type"] == "HAS_CONCEPT")
    for concept_id in sorted(concept_ids):
        entity = entity_by_id.get(concept_id)
        if entity:
            graph_edges.append({"source": concept_id, "target": f"class:{entity['class_id']}", "type": "INSTANCE_OF"})
            graph_edges.append({"source": f"ontology:{ontology['ontology_id']}", "target": f"class:{entity['class_id']}", "type": "HAS_CLASS"})

    for relation in ontology.get("relations", []):
        if relation["source_id"] in concept_ids and relation["target_id"] in concept_ids:
            graph_edges.append({"source": relation["source_id"], "target": relation["target_id"], "type": readable_relation_label(relation.get("label", "ONTOLOGY_RELATION"))})

    return unique_nodes(graph_nodes), unique_edges(graph_edges)


# SVG bleibt intern ein praktisches Exportformat, wird aber nur in HTML-Dateien eingebettet.
def build_graph_svg(graph_nodes: list[dict], graph_edges: list[dict], title: str, columns: list[tuple[str, str, int]]) -> tuple[str, dict[str, int]]:
    groups = {node_type: sorted([node for node in graph_nodes if node["type"] == node_type], key=lambda item: (-(item.get("score") or 0), item["label"])) for node_type, _, _ in columns}
    positioned = []
    for node_type, _label, x in columns:
        positioned.extend(distribute(groups.get(node_type, []), x, 145, 735))

    position_by_id = {node["id"]: node for node in positioned}
    colors = {
        "Ontology": "#0f766e",
        "OntologyClass": "#0369a1",
        "Entity": "#7c3aed",
        "Document": "#2563eb",
        "Chunk": "#16a34a",
        "NeighborChunk": "#84cc16",
        "GraphChunk": "#0d9488",
        "Term": "#f97316",
    }
    radii = {"Ontology": 19, "OntologyClass": 15, "Entity": 13, "Document": 19, "Chunk": 16, "GraphChunk": 13, "NeighborChunk": 10, "Term": 10}
    edge_colors = {
        "HAS_CLASS": "#0284c7",
        "INSTANCE_OF": "#38bdf8",
        "HAS_CHUNK": "#2563eb",
        "HAS_CONCEPT": "#7c3aed",
        "MENTIONS": "#f97316",
        "RELATED_1H": "#94a3b8",
        "GRAPH_EVIDENCE": "#0d9488",
    }

    svg_parts = [
        '<svg viewBox="0 0 1280 860" width="100%" height="860" role="img" aria-label="Graf-Kompass Graphansicht">',
        '<rect x="0" y="0" width="1280" height="860" fill="#f8fafc"/>',
        f'<text x="40" y="48" font-size="24" font-weight="800" fill="#0f172a">{escape(title)}</text>',
    ]

    for _node_type, label, x in columns:
        svg_parts.append(f'<text x="{x}" y="105" text-anchor="middle" font-size="14" font-weight="700" fill="#334155">{escape(label)}</text>')

    visible_edges = 0
    for edge in graph_edges:
        source = position_by_id.get(edge["source"])
        target = position_by_id.get(edge["target"])
        if not source or not target:
            continue
        visible_edges += 1
        color = edge_colors.get(edge["type"], "#64748b")
        width = 2.4 if edge["type"] in {"HAS_CHUNK", "HAS_CONCEPT", "HAS_CLASS", "INSTANCE_OF"} else 1.2
        svg_parts.append(
            f'<line x1="{source["x"]:.1f}" y1="{source["y"]:.1f}" x2="{target["x"]:.1f}" y2="{target["y"]:.1f}" '
            f'stroke="{color}" stroke-width="{width}" stroke-opacity="0.48"/>'
        )

    for node in positioned:
        color = colors.get(node["type"], "#64748b")
        radius = radii.get(node["type"], 12)
        title_text = escape(f"{node['label']} | {node['type']}")
        svg_parts.append(f'<g><title>{title_text}</title><circle cx="{node["x"]:.1f}" cy="{node["y"]:.1f}" r="{radius}" fill="{color}" stroke="white" stroke-width="2.4"/></g>')
        svg_parts.append(f'<text x="{node["x"]:.1f}" y="{node["y"] + radius + 13:.1f}" text-anchor="middle" font-size="10" fill="#0f172a">{escape(node["label_short"])}</text>')

    svg_parts.append("</svg>")
    summary = {label: len(groups.get(node_type, [])) for node_type, label, _x in columns}
    summary["Kanten"] = visible_edges
    return "\n".join(svg_parts), summary


def build_graph_html(title: str, lead: str, svg: str, summary: dict[str, int]) -> str:
    metrics = "".join(f'<div class="metric"><strong>{value}</strong><span>{escape(label)}</span></div>' for label, value in summary.items())
    return f"""<!doctype html>
<html lang="de">
<head>
  <meta charset="utf-8" />
  <title>{escape(title)}</title>
  <style>
    body {{ margin: 0; font-family: system-ui, sans-serif; background: #f8fafc; color: #0f172a; }}
    header {{ padding: 24px 30px 10px; border-top: 3px solid #0f766e; }}
    h1 {{ margin: 0 0 8px; font-size: 25px; }}
    p {{ margin: 0; color: #475569; max-width: 980px; line-height: 1.45; }}
    .metrics {{ display: flex; gap: 10px; flex-wrap: wrap; padding: 16px 30px 8px; }}
    .metric {{ min-width: 120px; padding: 10px 12px; background: white; border: 1px solid #e2e8f0; border-radius: 7px; box-shadow: 0 1px 2px rgba(15,23,42,.05); }}
    .metric strong {{ display: block; font-size: 20px; line-height: 1; }}
    .metric span {{ display: block; margin-top: 5px; font-size: 12px; color: #64748b; }}
    main {{ padding: 0 24px 28px; }}
    .graph {{ background: white; border: 1px solid #e2e8f0; border-radius: 8px; overflow: auto; box-shadow: 0 1px 3px rgba(15,23,42,.06); }}
  </style>
</head>
<body>
  <header>
    <h1>{escape(title)}</h1>
    <p>{escape(lead)}</p>
  </header>
  <section class="metrics">{metrics}</section>
  <main><section class="graph">{svg}</section></main>
</body>
</html>"""


document_nodes, document_edges = build_document_graph(nodes, edges)
entity_nodes, entity_edges = build_entity_graph(ONTOLOGY)
combined_nodes, combined_edges = build_combined_graph(nodes, edges, ONTOLOGY)

views = [
    (
        "document_graph.html",
        "Dokumentgraph",
        "Diese Ansicht zeigt Dokument, Top-Treffer, Nachbar-Chunks und einfache Begriffe. Sie ist gut, um Retrieval und Belegpfade zu prüfen.",
        document_nodes,
        document_edges,
        [("Document", "Dokument", 130), ("Chunk", "Top-Treffer", 360), ("GraphChunk", "Graph-Belege", 600), ("NeighborChunk", "1-Hop-Chunks", 820), ("Term", "Begriffe", 1060)],
    ),
    (
        "entity_graph.html",
        "Entity-Graph / Ontologiegraph",
        "Diese Ansicht zeigt die kuratierte Ontologie als Kontextkarte: Ontologie, Klassen, Entitäten und fachliche Beziehungen.",
        entity_nodes,
        entity_edges,
        [("Ontology", "Ontologie", 110), ("OntologyClass", "Klassen", 360), ("Entity", "Entitäten", 760)],
    ),
    (
        "combined_graph.html",
        "Kombinierter Graph",
        "Diese Ansicht verbindet Dokumentgraph und Ontologiegraph: Chunks zeigen auf gefundene Entitäten, Entitäten hängen an Klassen und fachlichen Nachbarschaften.",
        combined_nodes,
        combined_edges,
        [("Document", "Dokument", 80), ("Chunk", "Top-Treffer", 250), ("GraphChunk", "Graph-Belege", 430), ("Entity", "Entitäten", 620), ("OntologyClass", "Klassen", 835), ("Term", "Begriffe", 1050), ("NeighborChunk", "1-Hop", 1195)],
    ),
]

written_files = []
for filename, title, lead, graph_nodes, graph_edges, columns in views:
    svg, summary = build_graph_svg(graph_nodes, graph_edges, title, columns)
    html = build_graph_html(title, lead, svg, summary)
    path = OUTPUT_DIR / filename
    path.write_text(html, encoding="utf-8")
    written_files.append(path)

# Kompatibler Alias für ältere Doku und bisherige Erwartung.
(OUTPUT_DIR / "provenance.html").write_text((OUTPUT_DIR / "combined_graph.html").read_text(encoding="utf-8"), encoding="utf-8")
written_files.append(OUTPUT_DIR / "provenance.html")

show_json(
    "Graph-HTML-Dateien gespeichert",
    "Das Notebook rendert kein großes SVG mehr inline. Öffne die HTML-Dateien separat im Browser, damit die Notebook-Oberfläche lesbar bleibt.",
    {path.name: str(path) for path in written_files},
)

display(Markdown("\n".join(["### Graphansichten öffnen", ""] + [f"- `{path.name}`: `{path}`" for path in written_files])))


## 11. Retrieval-Brief mit Belegen

Dieser Abschnitt erzeugt eine knappe Markdown-Datei mit den wichtigsten Treffern, bevor das LLM daraus eine Antwort formuliert.

**Was der Code macht:** Er nimmt die besten Treffer aus der Vector Search, zeigt daraus lesbare Belege und schreibt sie als `retrieval_brief.md`.

**Was du danach sehen solltest:** Eine Bestätigung, wo `retrieval_brief.md` gespeichert wurde. Diese Datei ist noch keine freie LLM-Antwort, sondern der prüfbare Belegkontext für den nächsten Abschnitt.

In [ ]:
# Retrieval-Brief: Diese Datei zeigt fragebezogen, was das Retrieval tatsächlich gefunden hat.
SCENARIO_TITLES = {
    "klima_kommunale_klimaanpassung": "GraphRAG-Retrieval: Kommunale Klimaanpassung",
    "bsi_it_grundschutz": "GraphRAG-Retrieval: BSI IT-Grundschutz",
    "sozialbericht_2024": "GraphRAG-Retrieval: Sozialbericht 2024",
}

SCENARIO_DOMAIN_SIGNALS = {
    "klima_kommunale_klimaanpassung": ["klimaanpassung", "kommune", "kommunen", "maßnahme", "maßnahmen", "hebel", "resilienz", "hitze", "starkregen"],
    "bsi_it_grundschutz": ["anforderung", "anforderungen", "baustein", "bausteine", "schutzbedarf", "institution", "informationssicherheit", "grundschutz", "maßnahme"],
    "sozialbericht_2024": ["indikator", "indikatoren", "lebenslage", "lebenslagen", "armut", "bildung", "gesundheit", "einkommen", "bevölkerung", "sozial"],
}

SCENARIO_NEXT_STEPS = {
    "klima_kommunale_klimaanpassung": "Wenn die Belege zu allgemein bleiben, lohnt als nächstes ein feineres Chunking nach Abschnitten und eine strengere Entity-Linking-Logik für Maßnahmen, Risiken und Akteure.",
    "bsi_it_grundschutz": "Wenn die Belege zu allgemein bleiben, lohnt als nächstes eine stärkere Nutzung der XML-Struktur: Baustein, Anforderung, Rolle und Maßnahme sollten als eigene Knoten sichtbar werden.",
    "sozialbericht_2024": "Wenn die Belege zu breit bleiben, lohnt als nächstes ein Chunking nach Kapiteln, Tabellen und Indikatorabschnitten, damit Lebenslagen und Datenbefunde präziser werden.",
}


def classify_hit(row: dict) -> str:
    text = row["text"].lower()
    bibliography_markers = ["doi", "http", "www.", " in: ", " et al.", "s. ", "isbn", "literatur", "verfügbar"]
    marker_count = sum(text.count(marker) for marker in bibliography_markers)
    domain_signals = SCENARIO_DOMAIN_SIGNALS.get(SCENARIO_ID, [])
    has_domain_signal = any(signal in text for signal in domain_signals)
    if marker_count >= 4 and not has_domain_signal:
        return "eher Randkontext oder Literatur"
    if has_domain_signal:
        return "inhaltlich plausibler Treffer"
    return "prüfen"


def clean_snippet(text: str, width: int = 430) -> str:
    text = " ".join(text.replace("\n", " ").split())
    text = text.replace("- ", "")
    return textwrap.shorten(text, width=width, placeholder=" ...")


def evidence_title(row: dict, index: int) -> str:
    ontology_entities = _normalize_concepts(row.get("concepts", [])) if '_normalize_concepts' in globals() else []
    if ontology_entities:
        return f"Beleg {index}: {', '.join(ontology_entities[:2])}"
    terms = [term for term in row.get("terms", []) if len(term) > 3]
    if terms:
        return f"Beleg {index}: {', '.join(terms[:2]).title()}"
    return f"Beleg {index}"


def relevance_sentence(row: dict) -> str:
    ontology_entities = _normalize_concepts(row.get("concepts", [])) if '_normalize_concepts' in globals() else []
    ontology_neighbors = _normalize_related_concepts(row.get("related_concepts", [])) if '_normalize_related_concepts' in globals() else []
    if ontology_entities and ontology_neighbors:
        return f"Dieser Treffer ist relevant, weil er direkte Entitäten ({', '.join(ontology_entities[:3])}) und Graph-Nachbarn ({', '.join(ontology_neighbors[:3])}) verbindet."
    if ontology_entities:
        return f"Dieser Treffer ist relevant, weil er mit diesen Ontologie-Entitäten verbunden ist: {', '.join(ontology_entities[:4])}."
    return "Dieser Treffer ist relevant, weil er semantisch nah an der aktiven Frage liegt."


def graph_path_line(path: dict, index: int) -> str:
    if "format_graph_path" in globals():
        rendered = format_graph_path(path)
    else:
        rendered = " → ".join(node.get("name", "") for node in path.get("path_nodes", []) if node.get("name"))
    return f"- Pfad {index}: {rendered}"


def build_retrieval_brief(rows: list[dict], question: str, output_dir: Path) -> str:
    title = SCENARIO_TITLES.get(SCENARIO_ID, "GraphRAG-Retrieval")
    question_level = globals().get("ACTIVE_QUESTION_LEVEL", "unbekannt")
    relevant_rows = [row for row in rows if classify_hit(row) == "inhaltlich plausibler Treffer"]
    review_rows = [row for row in rows if classify_hit(row) != "inhaltlich plausibler Treffer"]
    lead_rows = relevant_rows[:4] or rows[:4]
    graph_paths_for_brief = globals().get("graph_paths", [])[:8]
    expanded_rows_for_brief = globals().get("expanded_rows", [])[:4]

    lines = [
        f"# {title}",
        "",
        "## Aktive Frage",
        "",
        question,
        "",
        "## Frage-Konfiguration",
        "",
        f"- Fragelevel: `{question_level}`",
        f"- TOP_K: `{TOP_K}`",
        f"- Graph-Hops: `{globals().get('GRAPH_HOPS', 'nicht ausgeführt')}`",
        "",
        "## Was dieses Briefing ist",
        "",
        "Dieses Dokument ist kein fertiger Fachaufsatz. Es zeigt, welche Belege das aktuelle Retrieval zur aktiven Frage gefunden hat und welche Graphpfade zusätzlichen Kontext öffnen. Wenn du die Frage änderst, sollte sich dieses Briefing entsprechend verändern.",
        "",
        "## Direkte Vector-Belege",
        "",
    ]

    for index, row in enumerate(lead_rows, 1):
        ontology_entities = _normalize_concepts(row.get("concepts", [])) if '_normalize_concepts' in globals() else []
        ontology_neighbors = _normalize_related_concepts(row.get("related_concepts", [])) if '_normalize_related_concepts' in globals() else []
        lines.extend([
            f"### {evidence_title(row, index)}",
            "",
            relevance_sentence(row),
            "",
            f"> {clean_snippet(row['text'])}",
            "",
            "Technische Spur:",
            f"- Quelle: `{row['doc_id']}`",
            f"- Chunk: `{row['chunk_id']}`",
            f"- Ähnlichkeit: `{row['score']:.3f}`",
            f"- Ontologie-Entitäten: {', '.join(ontology_entities) or 'keine'}",
            f"- Ontologie-Nachbarn: {', '.join(ontology_neighbors) or 'keine'}",
            f"- Einstufung: {classify_hit(row)}",
            "",
        ])

    lines.extend(["## Graphpfade", ""])
    if graph_paths_for_brief:
        lines.append("Diese Pfade kommen nicht aus dem Text allein, sondern aus der Ontologie-Nachbarschaft der gefundenen Entitäten.")
        lines.append("")
        lines.extend(graph_path_line(path, index) for index, path in enumerate(graph_paths_for_brief, 1))
        lines.append("")
    else:
        lines.extend(["Keine Graphpfade gefunden oder Abschnitt 8 wurde noch nicht ausgeführt.", ""])

    lines.extend(["## Ergänzende Graph-Belege", ""])
    if expanded_rows_for_brief:
        lines.append("Diese Chunks wurden über Pfad-Entitäten ergänzt. Sie sind nicht die primären Treffer, sondern zeigen, wohin der Graph das Retrieval erweitert.")
        lines.append("")
        for index, row in enumerate(expanded_rows_for_brief, 1):
            via = ", ".join(entity.get("name", "") for entity in row.get("via_entities", [])) or "keine"
            lines.extend([
                f"### Graph-Beleg {index}: {row['chunk_id']}",
                "",
                f"Über Entitäten: {via}",
                "",
                f"> {clean_snippet(row['text'])}",
                "",
            ])
    else:
        lines.extend(["Keine ergänzenden Graph-Belege gefunden oder Abschnitt 8 wurde noch nicht ausgeführt.", ""])

    if review_rows:
        lines.extend([
            "## Noch zu prüfen",
            "",
            "Diese Treffer sind semantisch nah, könnten aber Randkontext, Literaturhinweise oder noch unsauber geschnittene Chunks enthalten.",
            "",
        ])
        for row in review_rows[:3]:
            lines.append(f"- `{row['chunk_id']}` (`{row['score']:.3f}`): {clean_snippet(row['text'], width=260)}")
        lines.append("")

    lines.extend([
        "## Nächster sinnvoller Schritt",
        "",
        SCENARIO_NEXT_STEPS.get(SCENARIO_ID, "Als nächstes sollten Chunking, Entity Linking und Graph Expansion fachlich geprüft werden."),
        "",
        "## Dateien",
        "",
        f"- Provenienz-JSON: `{output_dir / 'provenance.json'}`",
        f"- Kombinierte Graphansicht: `{output_dir / 'combined_graph.html'}`",
        f"- Provenienz-HTML: `{output_dir / 'provenance.html'}`",
    ])
    return "\n".join(lines)

retrieval_brief = build_retrieval_brief(rows, QUESTION, OUTPUT_DIR)
(OUTPUT_DIR / "retrieval_brief.md").write_text(retrieval_brief, encoding="utf-8")
show_file(
    "Retrieval-Brief gespeichert",
    "Diese Markdown-Datei ist jetzt fragebezogen: Sie dokumentiert direkte Treffer, Graphpfade und ergänzende Graph-Belege zur aktiven Frage.",
    OUTPUT_DIR / "retrieval_brief.md",
)


## 12. Antwortvergleich: LLM vs. RAG vs. GraphRAG

Dieser Abschnitt macht den Nutzen des Retrieval-Patterns sichtbar, statt ihn nur zu behaupten.

**Was der Code macht:** Er erzeugt drei Antworten zur selben Frage: einmal ohne Kontext, einmal nur mit direkten Chunk-Belegen und einmal mit vollständigem GraphRAG-Kontext aus Chunks, Entitäten, Graphpfaden und ergänzenden Graph-Belegen. Danach vergleicht ein LLM die drei Antworten methodisch.

**Was du danach sehen solltest:** Vier Antwort-/Vergleichsdateien (`answer_llm_only.md`, `answer_rag_only.md`, `answer_graphrag.md`, `answer_comparison.md`) plus zwei Kontextdateien (`answer_rag_context.md`, `answer_graphrag_context.md`). Der Vergleich darf ausdrücklich zeigen, wenn RAG und GraphRAG bei einer einfachen Frage nah beieinander liegen.


In [ ]:
# Antwortvergleich: dieselbe Frage läuft durch drei unterschiedlich starke Kontext-Setups.
def context_row(row: dict, index: int) -> dict:
    concepts = _normalize_concepts(row.get("concepts", [])) if '_normalize_concepts' in globals() else []
    if not concepts:
        raw_concepts = row.get("concepts", [])
        if isinstance(raw_concepts, str):
            concepts = [item.strip() for item in raw_concepts.split(",") if item.strip()]
        elif isinstance(raw_concepts, (list, tuple, set)):
            concepts = [str(item).strip() for item in raw_concepts if str(item).strip()]

    terms = _normalize_list_value(row.get("terms", [])) if '_normalize_list_value' in globals() else []
    if not terms:
        raw_terms = row.get("terms", [])
        if isinstance(raw_terms, str):
            terms = [item.strip() for item in raw_terms.split(",") if item.strip()]
        elif isinstance(raw_terms, (list, tuple, set)):
            terms = [str(item).strip() for item in raw_terms if str(item).strip()]

    return {
        "beleg_nr": index,
        "chunk_id": row.get("chunk_id", ""),
        "quelle": row.get("doc_id", "unbekannt"),
        "aehnlichkeit": round(float(row.get("score", 0.0)), 3),
        "ontology_entities": concepts,
        "ontology_neighbors": _normalize_related_concepts(row.get("related_concepts", [])) if '_normalize_related_concepts' in globals() else [],
        "einfache_terms": terms,
        "text": clean_snippet(row.get("text", ""), width=1400),
    }


def rag_context_row(row: dict, index: int) -> dict:
    # RAG-only bekommt bewusst nur Textbelege, keine Entitäten und keine Graphpfade.
    return {
        "beleg_nr": index,
        "chunk_id": row.get("chunk_id", ""),
        "quelle": row.get("doc_id", "unbekannt"),
        "aehnlichkeit": round(float(row.get("score", 0.0)), 3),
        "text": clean_snippet(row.get("text", ""), width=1400),
    }


scenario_title = ONTOLOGY.get("title", SCENARIO_ID) if "ONTOLOGY" in globals() else SCENARIO_ID
llm_context = [context_row(row, index) for index, row in enumerate(rows[: min(TOP_K, 6)], 1)]
rag_context = [rag_context_row(row, index) for index, row in enumerate(rows[: min(TOP_K, 6)], 1)]
expanded_context = [
    context_row(row, index)
    for index, row in enumerate(globals().get("expanded_rows", [])[: globals().get("MAX_EXPANDED_CHUNKS", 6)], 1)
]
path_context = [
    {
        "pfad_nr": index,
        "start_entität": path.get("seed_name", ""),
        "ziel_entität": path.get("target_name", ""),
        "hops": path.get("hops", 0),
        "pfad": format_graph_path(path) if "format_graph_path" in globals() else path.get("path_nodes", []),
    }
    for index, path in enumerate(globals().get("graph_paths", [])[: globals().get("MAX_GRAPH_PATHS", 14)], 1)
]


def write_context_markdown(filename: str, title: str, direct_items: list[dict], include_graph: bool = False) -> Path:
    parts = [f"# {title}", "", "## Aktive Frage", "", QUESTION]
    parts.append("# Direkte Belege")
    for item in direct_items:
        lines = [
            f"## Beleg {item['beleg_nr']}: {item['chunk_id']}",
            f"Quelle: `{item['quelle']}`",
            f"Ähnlichkeit: `{item['aehnlichkeit']}`",
        ]
        if include_graph:
            lines.extend(
                [
                    f"Ontologie-Entitäten: {', '.join(item.get('ontology_entities', [])) or 'keine'}",
                    f"Ontologie-Nachbarn: {', '.join(item.get('ontology_neighbors', [])) or 'keine'}",
                ]
            )
        lines.extend(["", item["text"]])
        parts.append("\n".join(lines))

    if include_graph:
        parts.append("# Graphpfade")
        for item in path_context:
            parts.append(
                "\n".join(
                    [
                        f"## Pfad {item['pfad_nr']}: {item['start_entität']} → {item['ziel_entität']}",
                        f"Hops: `{item['hops']}`",
                        "",
                        str(item["pfad"]),
                    ]
                )
            )
        parts.append("# Ergänzende Graph-Belege")
        for item in expanded_context:
            parts.append(
                "\n".join(
                    [
                        f"## Graph-Beleg {item['beleg_nr']}: {item['chunk_id']}",
                        f"Quelle: `{item['quelle']}`",
                        f"Ontologie-Entitäten: {', '.join(item['ontology_entities']) or 'keine'}",
                        f"Ontologie-Nachbarn: {', '.join(item['ontology_neighbors']) or 'keine'}",
                        "",
                        item["text"],
                    ]
                )
            )

    path = OUTPUT_DIR / filename
    path.write_text("\n\n".join(parts) + "\n", encoding="utf-8")
    return path


rag_context_path = write_context_markdown("answer_rag_context.md", "RAG-only Kontext", rag_context, include_graph=False)
graphrag_context_path = write_context_markdown("answer_graphrag_context.md", "GraphRAG-Kontext", llm_context, include_graph=True)


def create_response_text(system_prompt: str, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    if LLM_PROVIDER == "github":
        try:
            response = client.chat.completions.create(
                model=ANSWER_MODEL,
                messages=messages,
                temperature=ANSWER_TEMPERATURE,
            )
        except Exception as exc:  # noqa: BLE001
            if "Unsupported parameter: 'temperature'" not in str(exc):
                raise
            response = client.chat.completions.create(
                model=ANSWER_MODEL,
                messages=messages,
            )
        return (response.choices[0].message.content or "").strip()

    # GPT-5-Modelle akzeptieren teils keine temperature; der Fallback ruft dann ohne diesen Parameter auf.
    try:
        response = client.responses.create(
            model=ANSWER_MODEL,
            input=messages,
            temperature=ANSWER_TEMPERATURE,
        )
    except Exception as exc:  # noqa: BLE001
        if "Unsupported parameter: 'temperature'" not in str(exc):
            raise
        response = client.responses.create(
            model=ANSWER_MODEL,
            input=messages,
        )
    return response.output_text.strip()


def answer_document(method_title: str, method_description: str, answer_text: str) -> str:
    return "\n".join(
        [
            f"# {method_title}: {scenario_title}",
            "",
            "## Aktive Frage",
            "",
            QUESTION,
            "",
            "## Methode",
            "",
            method_description,
            "",
            "## Frage-Konfiguration",
            "",
            f"- Fragelevel: `{ACTIVE_QUESTION_LEVEL}`",
            f"- Antwortprovider: `{LLM_PROVIDER}`",
            f"- Antwortmodell: `{ANSWER_MODEL}`",
            "",
            "## Antwort",
            "",
            answer_text,
            "",
        ]
    )


llm_only_system_prompt = """
Du bist ein deutschsprachiger Assistent für einen GraphRAG-Lernvergleich.
Du bekommst nur die Frage und keine Dokumentbelege.
Antworte hilfreich, aber kennzeichne klar, dass deine Antwort nicht aus den lokalen Projektquellen belegt ist.
Erfinde keine konkreten Quellenangaben.
""".strip()

llm_only_user_prompt = f"""
Frage:
{QUESTION}

Schreibe eine kurze Markdown-Antwort mit diesen Abschnitten:
1. Kurzantwort
2. Wahrscheinliche fachliche Einordnung
3. Unsicherheit ohne Quellenkontext
""".strip()

rag_system_prompt = """
Du bist ein deutschsprachiger RAG-Assistent für den Graf-Kompass.
Antworte nur auf Basis der gelieferten direkten Textbelege.
Nutze keine Ontologie, keine Graphpfade und kein externes Wissen.
Wenn die Belege für eine Aussage nicht reichen, benenne die Grenze ausdrücklich.
Verweise bei fachlichen Aussagen auf die Belegnummern.
""".strip()

rag_user_prompt = f"""
Frage:
{QUESTION}

Direkte Textbelege als JSON:
{json.dumps(rag_context, ensure_ascii=False, indent=2)}

Schreibe eine Markdown-Antwort mit diesen Abschnitten:
1. Kurzantwort
2. Belegte Aussagen
3. Grenzen der RAG-only-Antwort
4. Genutzte Belege
""".strip()

graphrag_system_prompt = """
Du bist ein deutschsprachiger GraphRAG-Assistent für den Graf-Kompass.
Antworte nur auf Basis der gelieferten Belege, Entitäten und Graphpfade.
Wenn die Belege für eine Aussage nicht reichen, benenne die Grenze ausdrücklich.
Nutze kurze, klare Abschnitte und verweise bei fachlichen Aussagen auf Belegnummern und Graphpfade.
""".strip()

graphrag_user_prompt = f"""
Frage:
{QUESTION}

Direkte Vector-Belege als JSON:
{json.dumps(llm_context, ensure_ascii=False, indent=2)}

Graphpfade als JSON:
{json.dumps(path_context, ensure_ascii=False, indent=2)}

Ergänzende Graph-Belege als JSON:
{json.dumps(expanded_context, ensure_ascii=False, indent=2)}

Schreibe eine Markdown-Antwort mit diesen Abschnitten:
1. Kurzantwort
2. Was der Graph zusätzlich sichtbar macht
3. Belegte Aussagen
4. Grenzen der GraphRAG-Antwort
5. Genutzte Belege und Graphpfade
""".strip()

llm_only_answer = create_response_text(llm_only_system_prompt, llm_only_user_prompt)
rag_only_answer = create_response_text(rag_system_prompt, rag_user_prompt)
graphrag_answer = create_response_text(graphrag_system_prompt, graphrag_user_prompt)

llm_only_document = answer_document(
    "LLM-only Antwort",
    "Das Modell bekommt nur die Frage. Diese Antwort zeigt, was ohne lokale Belege, Retrieval und Graphstruktur entsteht.",
    llm_only_answer,
)
rag_only_document = answer_document(
    "RAG-only Antwort",
    "Das Modell bekommt nur direkte Textbelege aus der Vector Search. Entitäten, Ontologie-Nachbarn und Graphpfade werden bewusst ausgeblendet.",
    rag_only_answer,
)
graphrag_document = answer_document(
    "GraphRAG Antwort",
    "Das Modell bekommt direkte Textbelege, Ontologie-Entitäten, Graphpfade und ergänzende Graph-Belege.",
    graphrag_answer,
)

llm_only_path = OUTPUT_DIR / "answer_llm_only.md"
rag_only_path = OUTPUT_DIR / "answer_rag_only.md"
graphrag_path = OUTPUT_DIR / "answer_graphrag.md"
llm_only_path.write_text(llm_only_document, encoding="utf-8")
rag_only_path.write_text(rag_only_document, encoding="utf-8")
graphrag_path.write_text(graphrag_document, encoding="utf-8")

comparison_system_prompt = """
Du bist ein deutschsprachiger GraphRAG-Didaktiker.
Vergleiche drei Antworten als Methodenartefakte: LLM-only, RAG-only und GraphRAG.
Bewerte nicht, welche Antwort dir stilistisch besser gefällt, sondern was die jeweilige Architektur sichtbar macht.
Sage ausdrücklich, wenn RAG-only und GraphRAG bei dieser Frage nur geringe Unterschiede zeigen.
""".strip()

comparison_user_prompt = f"""
Frage:
{QUESTION}

LLM-only Antwort:
{llm_only_document}

RAG-only Antwort:
{rag_only_document}

GraphRAG Antwort:
{graphrag_document}

Schreibe einen Markdown-Vergleich mit diesen Abschnitten:
1. Kurzfazit
2. Was LLM-only sichtbar macht
3. Was RAG-only verbessert
4. Was GraphRAG zusätzlich zeigt
5. Wo der Graph-Mehrwert stark, schwach oder noch nicht sichtbar ist
6. Didaktische Lektion für dieses Notebook
""".strip()

comparison_answer = create_response_text(comparison_system_prompt, comparison_user_prompt)
comparison_document = "\n".join(
    [
        f"# Antwortvergleich: {scenario_title}",
        "",
        "## Aktive Frage",
        "",
        QUESTION,
        "",
        "## Verglichene Methoden",
        "",
        "- LLM-only: nur Frage, kein lokaler Kontext",
        "- RAG-only: direkte Textbelege aus der Vector Search",
        "- GraphRAG: Textbelege plus Entitäten, Graphpfade und ergänzende Graph-Belege",
        "",
        "## Vergleich",
        "",
        comparison_answer,
        "",
    ]
)
comparison_path = OUTPUT_DIR / "answer_comparison.md"
comparison_path.write_text(comparison_document, encoding="utf-8")

display(Markdown(comparison_document))
show_json(
    "Antwortvergleich gespeichert",
    "Diese Dateien zeigen dieselbe Frage in drei Kontext-Setups und vergleichen die Unterschiede methodisch.",
    {
        "llm_only": str(llm_only_path),
        "rag_only": str(rag_only_path),
        "graphrag": str(graphrag_path),
        "comparison": str(comparison_path),
        "rag_context": str(rag_context_path),
        "graphrag_context": str(graphrag_context_path),
        "answer_model": ANSWER_MODEL,
    },
)


## 13. Mini-Experimente: Parameter fühlen

Dieser Abschnitt ist zum Spielen gedacht. Er verändert nichts automatisch, sondern zeigt dir, welche Stellschrauben du bewusst ausprobieren kannst.

**Was der Code macht:** Er zeigt kleine Experimente, die du nacheinander durchführen kannst: Frage ändern, `TOP_K` anpassen, Terms ignorieren, Ontologie-Nachbarn stärker beachten.

**Was du danach sehen solltest:** Eine kleine Experimentierliste. Wähle eine Zeile, ändere den genannten Parameter oben im Notebook oder in `.env`, starte den Kernel neu und führe ab Abschnitt 0 erneut aus.


In [ ]:
# Kleine Experimente machen das Pattern greifbar: jeweils eine Stellschraube ändern und die Wirkung beobachten.
experiment_steps = pd.DataFrame([
    {
        "experiment": "Frage enger stellen",
        "änderung": "DEFAULT_QUESTION oder die Szenario-Question in `.env` fokussieren",
        "worauf achten": "Werden weniger, aber fachlich klarere Ontologie-Entitäten getroffen?",
    },
    {
        "experiment": "Mehr/weniger Treffer",
        "änderung": "TOP_K zum Beispiel auf 4 oder 12 setzen",
        "worauf achten": "Wird die Antwort präziser oder kommt mehr Rauschen in den Belegkontext?",
    },
    {
        "experiment": "Terms kritisch lesen",
        "änderung": "In Abschnitt 7 `terms` und `ontology_entities` vergleichen",
        "worauf achten": "Welche Kontextsignale sind nur Wörter, welche sind echte Fachknoten?",
    },
    {
        "experiment": "Ontologie im Neo4j Browser öffnen",
        "änderung": "Die Cypher-Abfragen aus Abschnitt 6 ausführen",
        "worauf achten": "Ergibt die Kontextkarte ohne Dokument-Chunks schon fachlich Sinn?",
    },
])
show_table(
    "Experimente für das praktische Gefühl",
    "GraphRAG versteht man am besten, wenn man eine Stellschraube ändert und Retrieval, Graph und Antwort miteinander vergleicht.",
    experiment_steps,
)


## 14. Auswertung

Dieser letzte Abschnitt ist deine fachliche Bewertung nach dem Durchlauf.

Prüfe nach dem ersten Lauf:

- Verstehst du den Unterschied zwischen Dokumentgraph, Ontologiegraph, Retrievalgraph und Antwortsynthese?
- Ist der Top-Treffer fachlich plausibel?
- Erzählt der Provenienzgraph einen verständlichen Belegpfad?
- Sind die Ontologie-Entitäten hilfreicher als die einfachen Term-Knoten?
- Kannst du in Neo4j die Kontextkarte unabhängig von den Chunks nachvollziehen?
- Würde sich daraus eine Grafik-Lab-Demo mit Retrieval-, Ontologie- und LLM-Synthese-Pattern bauen lassen?
